# Cardiovascular arousal response strength is related to obstructive sleep apnea severity, impaired vigilance, and sleep depth
Code by Mieli Luukinen, licensed with the MIT License.

There are only few comments, with the main purpose of describing what needs to be done to replicate our results with your own data, rather than to explain what the code does. I have attempted to remove all code that isn't relevant to the final publication, but some remnants may still be there. There is also a slight risk that in that process I have removed something that is relevant, but I don't think so.

## Preparations

In [ ]:
%matplotlib widget

In [ ]:
# These supposedly limit the use of CPU cores, which may be useful on a shared server.

import os

NUM_THREADS = "16"

os.environ["OMP_NUM_THREADS"] = NUM_THREADS
os.environ["OPENBLAS_NUM_THREADS"] = NUM_THREADS
os.environ["MKL_NUM_THREADS"] = NUM_THREADS
os.environ["VECLIB_MAXIMUM_THREADS"] = NUM_THREADS
os.environ["NUMEXPR_NUM_THREADS"] = NUM_THREADS

In [ ]:
from os import listdir
from os.path import isfile, join
import csv
import datetime
import itertools
import json
import pickle

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.gridspec import GridSpec
import scipy.signal, scipy.stats, scipy.optimize, scipy.ndimage
from IPython.display import clear_output
from tqdm.notebook import tqdm, trange
from tqdm.contrib.concurrent import process_map
from tqdm import tqdm as tqdm_console
from statsmodels.stats.descriptivestats import sign_test
from statsmodels.discrete.discrete_model import Logit
import openpyxl
import pandas

from utils import *
from fit_analysis import *

In [ ]:
figfont8 = {'fontname':'Times New Roman', 'fontsize':8}
figfont10 = {'fontname':'Times New Roman', 'fontsize':10}
matplotlib.rcParams['mathtext.fontset'] = 'stix'

In [ ]:
# So, lots of data in different folders. You probably need to do some tweaking to use the code with your own data. Sadly, I don't have exact specifications for the formats our data are in, so you are
# mostly on your own. The main thing is that you need scored polysomnograms and psychomotor vigilance task data.

# Here the path for photoplethysmogram data is commented out due to some changes in our directories after I had extracted what I needed from them and saved it elsewhere. You should probably change that.

event_path = '/wrk/luukinen/data/event_files'
header_path = '/wrk/luukinen/data/headers/'
hypno_path = '/wrk/luukinen/data/hypnograms'
# pleth_path = '/wrk/rikuhut/data/stag/australia/formatted/Pleth'
pvt_path = '/wrk/luukinen/data/PVT_results'
headerstart = 'header_Compumedics ProFusion PSG - '
headerend = '.json'
eventstart = 'events_Compumedics ProFusion PSG - '
eventend = '.TXT'
hypnostart = 'hypnogram_Compumedics ProFusion PSG - '
hypnoend = '.TXT'
plethend = '_Pleth.npy'
pvtstart = 'pvt_'
pvtend = '.xlsx'
demographics_path = '/wrk/luukinen/patientdata.xlsx'

In [ ]:
# Again, PPG data commented out. Do what you need to.

event_files = [f for f in tqdm(listdir(event_path)) if isfile(join(event_path, f))]
header_files = [f for f in tqdm(listdir(header_path)) if isfile(join(header_path, f))]
hypnograms = [f for f in tqdm(listdir(hypno_path)) if isfile(join(hypno_path, f))]
# pleth_files = [f for f in tqdm(listdir(pleth_path)) if isfile(join(pleth_path, f))]
pvt_files = [f for f in tqdm(listdir(pvt_path)) if isfile(join(pvt_path, f))]
# A somewhat slow way of dealing with the fact that edf_files has a subfolder.
# It would be faster to sort the lists and discard the last one in that one.
# However, this is more robust if someone changes  the files.

# Those last comments are old, as seen from the fact the code no longer uses data in EDF format.

In [ ]:
fit_path = '/wrk/luukinen/results/20220426/'
patients_fit = [f for f in tqdm(listdir(fit_path)) if isfile(join(fit_path, f))]
fit_path_spont = '/wrk/luukinen/results/20221101_spont/'
patients_fit_spont = [f for f in tqdm(listdir(fit_path_spont)) if isfile(join(fit_path_spont, f))]

In [ ]:
# Matching files. Here, again, PPG files have been removed, this time without even leaving a comment. If/when you add them again, it's probably the best to put them last so you don't break indexing.
files = {hdr.split()[4][1:]: [header_path + '/' + hdr,
 event_path + '/' + eventstart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + eventend,
 hypno_path + '/' + hypnostart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + hypnoend,
 '',
 pvt_path + '/' + pvtstart + hdr.split()[4][1:] + pvtend]
 for hdr in tqdm(header_files)
 if (eventstart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + eventend) in event_files
 and (hypnostart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + hypnoend) in hypnograms
 and pvtstart + hdr.split()[4][1:] + pvtend in pvt_files
 and (hdr.split()[4][1:] in patients_fit or hdr.split()[4][1:] in patients_fit_spont)
}

In [ ]:
patient_ids = sorted(list(files))

In [ ]:
wb = openpyxl.load_workbook('/wrk/luukinen/patientdata.xlsx', data_only=True)
ws = wb.active
demographics = []
for row in ws.values:
    demographics.append([])
    for value in row:
        demographics[-1].append(value)

In [ ]:
demographics_by_patient = {str(demographics[i][0]): demographics[i] for i in range(len(demographics)) if isinstance(demographics[i][0], int) and str(demographics[i][0]) in patient_ids}

In [ ]:
len([v for v in demographics_by_patient.values() if isinstance(v[3], int) or isinstance(v[3], float)])
exclude_no_demographics = []
for patient in patient_ids:
    v = demographics_by_patient[patient]
    if not (v[1] == 0 or v[1] == 1) or not (isinstance(v[2], int) or isinstance(v[2], float)) or not (isinstance(v[3], int) or isinstance(v[3], float)) or v[3] == 0:
        exclude_no_demographics.append(patient)
print(len(exclude_no_demographics))

# NOTE: This is untrue, check the number from previous paper!
# ANOTHER NOTE, MUCH LATER: I assume the untrueness mentioned in the above note (this number is supposed to tell how many subjects were excluded for not having demographic data) is again due to the
# missing PPG data files.

In [ ]:
pvt_rts_by_patient = {}
for patient in tqdm(patient_ids):
    pvt_rts_by_patient[patient] = pandas.read_excel(files[patient][4])['rt'].to_numpy()

In [ ]:
# This data is desaturation severity (DesSev) for each subject, calculated elsewhere.

wb = openpyxl.load_workbook('/wrk/luukinen/DesSev_table.xlsx', data_only=True)
ws = wb.active
dessevs_ = []
for row in ws.values:
    dessevs_.append([])
    for value in row:
        dessevs_[-1].append(value)

In [ ]:
dessevs = {str(dessevs_[i][0]): dessevs_[i][1] for i in range(len(dessevs_)) if isinstance(dessevs_[i][0], int) and str(dessevs_[i][0]) in patient_ids}

In [ ]:
exclude_no_dessevs = []
for patient in patient_ids:
    if not patient in dessevs.keys() and not patient in exclude_no_demographics:
        exclude_no_dessevs.append(patient)
print(len(exclude_no_dessevs))

## Compiling arousal information

In [ ]:
n_patients = np.inf  # A maximum number. Actual amount may be less.
times = {}
durations = {}
stages = {}
stages2 = {}
epochs = {}
sats = {}
desats = {}
arousal_inds = {} # respiratory arousals taken into analyses
arousal_inds_resp_all = {}
arousal_inds_spont = {}
arousal_inds_all = {}
hypopnea_inds = {}
apnea_inds = {}
central_apnea_inds = {}
mixed_apnea_inds = {}
related_events = {}
pleths = {}

if n_patients < len(patient_ids):
    patients = list(np.random.choice(patient_ids, n_patients, replace=False))
else:
    patients = patient_ids.copy()
for patient in exclude_no_demographics:
    if patient in patients:
        patients.remove(patient)
for patient in exclude_no_dessevs:
    if patient in patients:
        patients.remove(patient)

for patient in tqdm(patients):
    filepath = files[patient][0]
    with open(filepath, newline='') as f:
        header = json.load(f)
    start = datetime.datetime.fromisoformat(header['startdate'])
    sample_rate_original = [signal_header for signal_header in header['SignalHeaders'] if signal_header['label'] == 'Pleth'][0]['sample_rate']
    filepath = files[patient][1]
    with open(filepath, newline='') as f:
        reader = csv.reader(f)
        events = list(reader)
    # ISO format:
    for event in events:
        if event[0][1] == ':':
            event[0] = '0'+event[0]
    filepath = files[patient][2]
    with open(filepath, newline='') as f:
        stages[patient] = [stage2num(str.strip('\r\n')) for str in f.readlines()]

    date = start.date()  # incorrect after midnight, but works for the substractions
    _times = [datetime.datetime.combine(date, datetime.time.fromisoformat(event[0])) for event in events]
    times[patient] = [(time-start).total_seconds() % (60*60*24) for time in _times]
    durations[patient] = [duration_from_str(event[4]) for event in events]
    stages2[patient] = [stage2num(event[2], ['AWAKE', 'REM', 'N1', 'N2', 'N3']) for event in events]
    epochs[patient] = [int(event[1])-1 for event in events]
    sats[patient] = [int(event[5]) if event[5] != '-' else 0 for event in events]
    desats[patient] = [int(event[6]) if event[6] != '-' else 0 for event in events]

    arousal_inds[patient] = [i for i in np.where([event[3] == 'Arousal 2 ARO RES' for event in events])[0] if 3 < durations[patient][i] < 15 and times[patient][i] + durations[patient][i] < len(stages[patient])*30]
    arousal_inds_resp_all[patient] = [i for i in np.where([event[3] == 'Arousal 2 ARO RES' for event in events])[0]]
    arousal_inds_spont[patient] = [i for i in np.where([event[3] == 'Arousal 1 ARO SPONT' for event in events])[0] if 3 < durations[patient][i] < 15 and times[patient][i] + durations[patient][i] < len(stages[patient])*30]
    arousal_inds_all[patient] = [i for i in np.where([event[3][0:7] == 'Arousal' for event in events])[0]]
    hypopnea_inds[patient] = np.where([event[3] == 'Hypopnea' for event in events])[0]
    apnea_inds[patient] = np.where([event[3] == 'Obstructive Apnea' for event in events])[0]
    central_apnea_inds[patient] = np.where([event[3] == 'Central Apnea' for event in events])[0]
    mixed_apnea_inds[patient] = np.where([event[3] == 'Mixed Apnea' for event in events])[0]
    related_events[patient] = []
    for i in arousal_inds[patient]:
        e = []
        for j in hypopnea_inds[patient]:
            if times[patient][i] - 5 < times[patient][j] + durations[patient][j] < times[patient][i] + durations[patient][i]:
                e.append(j)
        for j in apnea_inds[patient]:
            if times[patient][i] - 5 < times[patient][j] + durations[patient][j] < times[patient][i] + durations[patient][i]:
                e.append(j)
        related_events[patient].append(sorted(e))
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(arousal_inds[patient])) if len(related_events[patient][i]) == 1]
    related_events[patient] = [e[0] for e in related_events[patient] if len(e) == 1]
    for i in arousal_inds[patient]:
        if stages2[patient][i] > -1:
            stages2[patient][i] = stages[patient][epochs[patient][i]-1]
    for i in arousal_inds_spont[patient]:
        if stages2[patient][i] > -1:
            stages2[patient][i] = stages[patient][epochs[patient][i]-1]
    related_events[patient] = [related_events[patient][i] for i in range(len(related_events[patient])) if stages2[patient][arousal_inds[patient][i]] < 0]
    arousal_inds[patient] = [i for i in arousal_inds[patient] if stages2[patient][i] < 0]
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(arousal_inds[patient])) if 0 <= sats[patient][related_events[patient][i]] <= 100]
    related_events[patient] = [e for e in related_events[patient] if 0 <= sats[patient][e] <= 100]
    arousal_inds_spont[patient] = [i for i in arousal_inds_spont[patient] if stages2[patient][i] < 0]
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(related_events[patient])) if durations[patient][related_events[patient][i]] > 10]
    related_events[patient] = [i for i in related_events[patient] if durations[patient][i] > 10]
    # pleths[patient] = scipy.signal.decimate(np.load(files[patient][3]), int(sample_rate_original/sample_rate))

In [ ]:
arousals_with_5s5s_margins = {}
arousals_with_10s5s_margins = {}
arousals_with_5s10s_margins = {}
arousals_with_10s10s_margins = {}
for patient in tqdm(patients):
    arousals_with_5s5s_margins[patient] = []
    arousals_with_10s5s_margins[patient] = []
    arousals_with_5s10s_margins[patient] = []
    arousals_with_10s10s_margins[patient] = []
    for i in range(len(arousal_inds[patient])):
        start = times[patient][arousal_inds[patient][i]]
        end = start + durations[patient][arousal_inds[patient][i]]
        within55 = False
        within105 = False
        within510 = False
        within1010 = False
        for j in np.concatenate((arousal_inds_all[patient], apnea_inds[patient], hypopnea_inds[patient], central_apnea_inds[patient], mixed_apnea_inds[patient])):
            if j != arousal_inds[patient][i] and j != related_events[patient][i]:
                if start - 5 < times[patient][j] < end + 5 or start - 5 < times[patient][j] + durations[patient][j] < end + 5:
                    within55 = True
                if start - 10 < times[patient][j] < end + 5 or start - 10 < times[patient][j] + durations[patient][j] < end + 5:
                    within105 = True
                if start - 5 < times[patient][j] < end + 10 or start - 5 < times[patient][j] + durations[patient][j] < end + 10:
                    within510 = True
                if start - 10 < times[patient][j] < end + 10 or start - 10 < times[patient][j] + durations[patient][j] < end + 10:
                    within1010 = True
        if not within55:
            arousals_with_5s5s_margins[patient].append(i)
        if not within105:
            arousals_with_10s5s_margins[patient].append(i)
        if not within510:
            arousals_with_5s10s_margins[patient].append(i)
        if not within1010:
            arousals_with_10s10s_margins[patient].append(i)
print(sum([len(arousal_inds[patient]) for patient in patients]))
print(sum([len(arousals_with_5s5s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_10s5s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_5s10s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]))

In [ ]:
arousals_with_10s10s_margins_spont = {}
for patient in tqdm(patients):
    arousals_with_10s10s_margins_spont[patient] = []
    for i in range(len(arousal_inds_spont[patient])):
        start = times[patient][arousal_inds_spont[patient][i]]
        end = start + durations[patient][arousal_inds_spont[patient][i]]
        within1010 = False
        for j in np.concatenate((arousal_inds_all[patient], apnea_inds[patient], hypopnea_inds[patient], central_apnea_inds[patient], mixed_apnea_inds[patient])):
            if j != arousal_inds_spont[patient][i]:
                if start - 10 < times[patient][j] < end + 10 or start - 10 < times[patient][j] + durations[patient][j] < end + 10:
                    within1010 = True
        if not within1010:
            arousals_with_10s10s_margins_spont[patient].append(i)
print(sum([len(arousal_inds_spont[patient]) for patient in patients]))
print(sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients]))

In [ ]:
print(len(patients))
patients = [patient for patient in patients if len(arousals_with_10s10s_margins[patient]) > 0 or len(arousals_with_10s10s_margins_spont[patient]) > 0]
print(len(patients))

## Three-step fitting model

In [ ]:
# The thing with this and the next three cells is that the fitting of the model takes ridiculously long. Like, in the order of a week or multiple for about 1000 subjects, if the code is running
# continuously. So, the fittings are saved after creating them, and loaded on the subsequent runs. This even works with random interruptions of the process, you can just continue from where you got.

frequency_popts = {}
amplitude_popts = {}
for patient in tqdm(patients_fit):
    if patient in patients:
        with open(join(fit_path, patient)) as resultfile:
            results = json.load(resultfile)
        for arousal in arousals_with_10s10s_margins[patient]:
            if not arousal in [result[0] for result in results]:
                raise Exception("Missing arousals for patient " + patient)
        frequency_popts[patient] = []
        amplitude_popts[patient] = []
        for result in results:
            if result[0] in arousals_with_10s10s_margins[patient]:
                frequency_popts[patient].append(result[1])
                amplitude_popts[patient].append(result[2])

In [ ]:
# frequency_popts = {}
# amplitude_popts = {}
for patient in tqdm([patient for patient in patients if not patient in frequency_popts.keys()]):
    pleth_time = np.arange(len(pleths[patient])) / sample_rate
    arousal_pleth_inds = [np.where((times[patient][i] < pleth_time) & (pleth_time < times[patient][i] + durations[patient][i]))[0] for i in arousal_inds[patient]]
    frequency_popts[patient] = []
    amplitude_popts[patient] = []
    for arousal_i in arousals_with_10s10s_margins[patient]:
        inds = arousal_pleth_inds[arousal_i]
        inds = np.arange(min(inds)-sample_rate*before_arousal, max(inds)+1+sample_rate*before_arousal)
        if inds[-1] < len(pleths[patient]) and inds[0] > 0:
            f, t, Sxx = scipy.signal.spectrogram(pleths[patient][inds], fs=sample_rate, nperseg=sample_rate*2, nfft=sample_rate*16, noverlap=sample_rate*2-1, window='boxcar')
            t = t - before_arousal

            ydata_ = np.average(np.tile(f, (len(t), 1)).transpose(), axis=0, weights=Sxx)
            ydata = scipy.signal.savgol_filter(ydata_, 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7 # If frequency or amplitude is negative at any point, something has gone wrong in the fitting. This is extremely rare.
            frequency_popts[patient].append(popt)

            ydata_ = scipy.ndimage.maximum_filter(pleths[patient][inds], size=sample_rate*2) - scipy.ndimage.minimum_filter(pleths[patient][inds], size=sample_rate*2)
            ydata = scipy.signal.savgol_filter(ydata_[sample_rate:-sample_rate+1], 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7
            amplitude_popts[patient].append(popt)

        else:
            popt = np.array([np.nan]*7)
            frequency_popts[patient].append(popt)
            amplitude_popts[patient].append(popt)
    with open(fit_path + patient, 'w') as resultfile:
        json.dump([[arousals_with_10s10s_margins[patient][i],  
                    list(frequency_popts[patient][i]),
                    list(amplitude_popts[patient][i])] for i in range(len(arousals_with_10s10s_margins[patient]))], resultfile)

In [ ]:
frequency_popts_spont = {}
amplitude_popts_spont = {}
for patient in tqdm(patients_fit_spont):
    if patient in patients:
        with open(join(fit_path_spont, patient)) as resultfile:
            results = json.load(resultfile)
        for arousal in arousals_with_10s10s_margins_spont[patient]:
            if not arousal in [result[0] for result in results]:
                raise Exception("Missing arousals for patient " + patient)
        frequency_popts_spont[patient] = []
        amplitude_popts_spont[patient] = []
        for result in results:
            if result[0] in arousals_with_10s10s_margins_spont[patient]:
                frequency_popts_spont[patient].append(result[1])
                amplitude_popts_spont[patient].append(result[2])

In [ ]:
for patient in tqdm([patient for patient in patients if not patient in frequency_popts_spont.keys()]):
    pleth_time = np.arange(len(pleths[patient])) / sample_rate
    arousal_pleth_inds = [np.where((times[patient][i] < pleth_time) & (pleth_time < times[patient][i] + durations[patient][i]))[0] for i in arousal_inds_spont[patient]]
    frequency_popts_spont[patient] = []
    amplitude_popts_spont[patient] = []
    for arousal_i in arousals_with_10s10s_margins_spont[patient]:
        inds = arousal_pleth_inds[arousal_i]
        inds = np.arange(min(inds)-sample_rate*before_arousal, max(inds)+1+sample_rate*before_arousal)
        if inds[-1] < len(pleths[patient]) and inds[0] > 0:
            f, t, Sxx = scipy.signal.spectrogram(pleths[patient][inds], fs=sample_rate, nperseg=sample_rate*2, nfft=sample_rate*16, noverlap=sample_rate*2-1, window='boxcar')
            t = t - before_arousal

            ydata_ = np.average(np.tile(f, (len(t), 1)).transpose(), axis=0, weights=Sxx)
            ydata = scipy.signal.savgol_filter(ydata_, 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7
            frequency_popts_spont[patient].append(popt)

            ydata_ = scipy.ndimage.maximum_filter(pleths[patient][inds], size=sample_rate*2) - scipy.ndimage.minimum_filter(pleths[patient][inds], size=sample_rate*2)
            ydata = scipy.signal.savgol_filter(ydata_[sample_rate:-sample_rate+1], 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
            xdata = t[sample_rate:-sample_rate+1]
            scale = max(ydata)

            try:
                popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
            except:
                popt = [np.nan]*7
            for startguess in [-5, 0, 5]:
                for midguess in [0.3, 0.5, 0.7]:
                    for endguess in [0.5, 0.7, 0.9]:
                        try:
                            _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                            if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                                popt = _popt
                        except:
                            pass
            if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
                popt = [np.nan]*7
            amplitude_popts_spont[patient].append(popt)

        else:
            popt = np.array([np.nan]*7)
            frequency_popts_spont[patient].append(popt)
            amplitude_popts_spont[patient].append(popt)
    with open(fit_path_spont + patient, 'w') as resultfile:
        json.dump([[arousals_with_10s10s_margins_spont[patient][i],  
                    list(frequency_popts_spont[patient][i]),
                    list(amplitude_popts_spont[patient][i])] for i in range(len(arousals_with_10s10s_margins_spont[patient]))], resultfile)

In [ ]:
arousal_durations = []
amp_classifications = []
amp_startbaselines = []
amp_endbaselines = []
amp_expected_changes = []
amp_unexpected_changes = []
amp_second_expected_changes = []
amp_second_unexpected_changes = []
amp_meanfitvalues = []
freq_classifications = []
freq_startbaselines = []
freq_endbaselines = []
freq_expected_changes = []
freq_unexpected_changes = []
freq_second_expected_changes = []
freq_second_unexpected_changes = []
freq_meanfitvalues = []
labels = []
for patient in tqdm(arousals_with_10s10s_margins.keys()):
    for i in range(len(arousals_with_10s10s_margins[patient])):
        amp_fit = amplitude_popts[patient][i]
        freq_fit = frequency_popts[patient][i]
        duration = durations[patient][arousal_inds[patient][arousals_with_10s10s_margins[patient][i]]]
        arousal_durations.append(duration)
        amp_classification = classify(amp_fit, False, duration)
        amp_classifications.append(amp_classification)
        amp_startbaselines.append(startbaseline(amp_fit, duration, amp_classification))
        amp_endbaselines.append(endbaseline(amp_fit, duration, amp_classification))
        amp_expected_changes.append(expected_change(amp_fit, duration, amp_classification))
        amp_unexpected_changes.append(unexpected_change(amp_fit, duration, amp_classification))
        amp_second_expected_changes.append(second_expected_change(amp_fit, duration, amp_classification))
        amp_second_unexpected_changes.append(second_unexpected_change(amp_fit, duration, amp_classification))
        amp_meanfitvalues.append(meanfitvalue(amp_fit, duration))
        freq_classification = classify(freq_fit, True, duration)
        freq_classifications.append(freq_classification)
        freq_startbaselines.append(startbaseline(freq_fit, duration, freq_classification))
        freq_endbaselines.append(endbaseline(freq_fit, duration, freq_classification))
        freq_expected_changes.append(expected_change(freq_fit, duration, freq_classification))
        freq_unexpected_changes.append(unexpected_change(freq_fit, duration, freq_classification))
        freq_second_expected_changes.append(second_expected_change(freq_fit, duration, freq_classification))
        freq_second_unexpected_changes.append(second_unexpected_change(freq_fit, duration, freq_classification))
        freq_meanfitvalues.append(meanfitvalue(freq_fit, duration))
        labels.append((patient, i))

In [ ]:
amp_eventual_expected_changes = []
amp_second_chosen = []
freq_eventual_expected_changes = []
freq_second_chosen = []
for i in tqdm(range(len(amp_expected_changes))):
    if (not np.isnan(amp_second_expected_changes[i][0])) and abs(amp_second_expected_changes[i][0] - amp_startbaselines[i])*(arousal_durations[i]+before_arousal-amp_second_expected_changes[i][1]) > abs(amp_expected_changes[i][0] - amp_startbaselines[i])*(amp_expected_changes[i][2]-amp_expected_changes[i][1]):
        amp_eventual_expected_changes.append(amp_second_expected_changes[i])
        amp_second_chosen.append(True)
    else:
        amp_eventual_expected_changes.append(amp_expected_changes[i])
        amp_second_chosen.append(False)
    if (not np.isnan(freq_second_expected_changes[i][0])) and abs(freq_second_expected_changes[i][0] - freq_startbaselines[i])*(arousal_durations[i]+before_arousal-freq_second_expected_changes[i][1]) > abs(freq_expected_changes[i][0] - freq_startbaselines[i])*(freq_expected_changes[i][2]-freq_expected_changes[i][1]):
        freq_eventual_expected_changes.append(freq_second_expected_changes[i])
        freq_second_chosen.append(True)
    else:
        freq_eventual_expected_changes.append(freq_expected_changes[i])
        freq_second_chosen.append(False)
print(np.count_nonzero(amp_second_chosen))
print(np.count_nonzero(freq_second_chosen))

In [ ]:
arousal_durations_spont = []
amp_classifications_spont = []
amp_startbaselines_spont = []
amp_endbaselines_spont = []
amp_expected_changes_spont = []
amp_unexpected_changes_spont = []
amp_second_expected_changes_spont = []
amp_second_unexpected_changes_spont = []
amp_meanfitvalues_spont = []
freq_classifications_spont = []
freq_startbaselines_spont = []
freq_endbaselines_spont = []
freq_expected_changes_spont = []
freq_unexpected_changes_spont = []
freq_second_expected_changes_spont = []
freq_second_unexpected_changes_spont = []
freq_meanfitvalues_spont = []
labels_spont = []
for patient in tqdm(arousals_with_10s10s_margins_spont.keys()):
    for i in range(len(arousals_with_10s10s_margins_spont[patient])):
        amp_fit = amplitude_popts_spont[patient][i]
        freq_fit = frequency_popts_spont[patient][i]
        duration = durations[patient][arousal_inds_spont[patient][arousals_with_10s10s_margins_spont[patient][i]]]
        arousal_durations_spont.append(duration)
        amp_classification = classify(amp_fit, False, duration)
        amp_classifications_spont.append(amp_classification)
        amp_startbaselines_spont.append(startbaseline(amp_fit, duration, amp_classification))
        amp_endbaselines_spont.append(endbaseline(amp_fit, duration, amp_classification))
        amp_expected_changes_spont.append(expected_change(amp_fit, duration, amp_classification))
        amp_unexpected_changes_spont.append(unexpected_change(amp_fit, duration, amp_classification))
        amp_second_expected_changes_spont.append(second_expected_change(amp_fit, duration, amp_classification))
        amp_second_unexpected_changes_spont.append(second_unexpected_change(amp_fit, duration, amp_classification))
        amp_meanfitvalues_spont.append(meanfitvalue(amp_fit, duration))
        freq_classification = classify(freq_fit, True, duration)
        freq_classifications_spont.append(freq_classification)
        freq_startbaselines_spont.append(startbaseline(freq_fit, duration, freq_classification))
        freq_endbaselines_spont.append(endbaseline(freq_fit, duration, freq_classification))
        freq_expected_changes_spont.append(expected_change(freq_fit, duration, freq_classification))
        freq_unexpected_changes_spont.append(unexpected_change(freq_fit, duration, freq_classification))
        freq_second_expected_changes_spont.append(second_expected_change(freq_fit, duration, freq_classification))
        freq_second_unexpected_changes_spont.append(second_unexpected_change(freq_fit, duration, freq_classification))
        freq_meanfitvalues_spont.append(meanfitvalue(freq_fit, duration))
        labels_spont.append((patient, i))

In [ ]:
amp_eventual_expected_changes_spont = []
amp_second_chosen_spont = []
freq_eventual_expected_changes_spont = []
freq_second_chosen_spont = []
for i in tqdm(range(len(amp_expected_changes_spont))):
    if (not np.isnan(amp_second_expected_changes_spont[i][0])) and abs(amp_second_expected_changes_spont[i][0] - amp_startbaselines_spont[i])*(arousal_durations_spont[i]+before_arousal-amp_second_expected_changes_spont[i][1]) > abs(amp_expected_changes_spont[i][0] - amp_startbaselines_spont[i])*(amp_expected_changes_spont[i][2]-amp_expected_changes_spont[i][1]):
        amp_eventual_expected_changes_spont.append(amp_second_expected_changes_spont[i])
        amp_second_chosen_spont.append(True)
    else:
        amp_eventual_expected_changes_spont.append(amp_expected_changes_spont[i])
        amp_second_chosen_spont.append(False)
    if (not np.isnan(freq_second_expected_changes_spont[i][0])) and abs(freq_second_expected_changes_spont[i][0] - freq_startbaselines_spont[i])*(arousal_durations_spont[i]+before_arousal-freq_second_expected_changes_spont[i][1]) > abs(freq_expected_changes_spont[i][0] - freq_startbaselines_spont[i])*(freq_expected_changes_spont[i][2]-freq_expected_changes_spont[i][1]):
        freq_eventual_expected_changes_spont.append(freq_second_expected_changes_spont[i])
        freq_second_chosen_spont.append(True)
    else:
        freq_eventual_expected_changes_spont.append(freq_expected_changes_spont[i])
        freq_second_chosen_spont.append(False)
print(np.count_nonzero(amp_second_chosen_spont))
print(np.count_nonzero(freq_second_chosen_spont))

In [ ]:
print('\nRESPIRATORY AROUSALS')
print('\nAmplitude:')
print(repr(len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) / len(amp_eventual_expected_changes) * 100) + ' % of the arousals showed a negative change in amplitude (as expected).')
print(repr(len([change for change in amp_eventual_expected_changes if change[4] == 1]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by an increase.')
print(repr(len([i for i in range(len(amp_eventual_expected_changes)) if amp_eventual_expected_changes[i][4] == -1 or amp_second_unexpected_changes[i][4] == 1]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by an increase.')
print(repr(len([i for i in range(len(amp_expected_changes)) if not np.isnan(amp_expected_changes[i][0]) and amp_second_chosen[i]]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by another, "smaller" decrease.')
print(repr(len([i for i in range(len(amp_expected_changes)) if not np.isnan(amp_expected_changes[i][0]) and not np.isnan(amp_second_expected_changes[i][0]) and not amp_second_chosen[i]]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by another, "smaller" decrease.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in amp_eventual_expected_changes if not np.isnan(change[3])]) / len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these decreases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(amp_expected_changes)) if np.isnan(amp_expected_changes[i][0]) and not np.isnan(amp_unexpected_changes[i][0])]) / len(amp_expected_changes) * 100) + ' % of the arousals showed an unaccompanied positive change (or two).')
print('\nFrequency:')
print(repr(len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) / len(freq_eventual_expected_changes) * 100) + ' % of the arousals showed a positive change in frequency (as expected).')
print(repr(len([change for change in freq_eventual_expected_changes if change[4] == 1]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by a decrease.')
print(repr(len([i for i in range(len(freq_eventual_expected_changes)) if freq_eventual_expected_changes[i][4] == -1 or freq_second_unexpected_changes[i][4] == 1]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by a decrease.')
print(repr(len([i for i in range(len(freq_expected_changes)) if not np.isnan(freq_expected_changes[i][0]) and freq_second_chosen[i]]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by another, "smaller" increase.')
print(repr(len([i for i in range(len(freq_expected_changes)) if not np.isnan(freq_expected_changes[i][0]) and not np.isnan(freq_second_expected_changes[i][0]) and not freq_second_chosen[i]]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by another, "smaller" increase.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in freq_eventual_expected_changes if not np.isnan(change[3])]) / len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) * 100) + ' % of these increases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(freq_expected_changes)) if np.isnan(freq_expected_changes[i][0]) and not np.isnan(freq_unexpected_changes[i][0])]) / len(freq_expected_changes) * 100) + ' % of the arousals showed an unaccompanied negative change (or two).')

print('\n' + repr(len([i for i in range(len(amp_eventual_expected_changes)) if not np.isnan(amp_eventual_expected_changes[i][0]) or not np.isnan(freq_eventual_expected_changes[i][0])]) / len(amp_eventual_expected_changes) * 100) + ' % of the arousals showed at least one type of expected change.')

In [ ]:
print('\nSPONTANEOUS AROUSALS')
print('\nAmplitude:')
print(repr(len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) / len(amp_eventual_expected_changes_spont) * 100) + ' % of the arousals showed a negative change in amplitude (as expected).')
print(repr(len([change for change in amp_eventual_expected_changes_spont if change[4] == 1]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by an increase.')
print(repr(len([i for i in range(len(amp_eventual_expected_changes_spont)) if amp_eventual_expected_changes_spont[i][4] == -1 or amp_second_unexpected_changes_spont[i][4] == 1]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by an increase.')
print(repr(len([i for i in range(len(amp_expected_changes_spont)) if not np.isnan(amp_expected_changes_spont[i][0]) and amp_second_chosen_spont[i]]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were preceded by another, "smaller" decrease.')
print(repr(len([i for i in range(len(amp_expected_changes_spont)) if not np.isnan(amp_expected_changes_spont[i][0]) and not np.isnan(amp_second_expected_changes_spont[i][0]) and not amp_second_chosen_spont[i]]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases were followed by another, "smaller" decrease.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[3])]) / len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these decreases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(amp_expected_changes_spont)) if np.isnan(amp_expected_changes_spont[i][0]) and not np.isnan(amp_unexpected_changes_spont[i][0])]) / len(amp_expected_changes_spont) * 100) + ' % of the arousals showed an unaccompanied positive change (or two).')
print('\nFrequency:')
print(repr(len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) / len(freq_eventual_expected_changes_spont) * 100) + ' % of the arousals showed a positive change in frequency (as expected).')
print(repr(len([change for change in freq_eventual_expected_changes_spont if change[4] == 1]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by a decrease.')
print(repr(len([i for i in range(len(freq_eventual_expected_changes_spont)) if freq_eventual_expected_changes_spont[i][4] == -1 or freq_second_unexpected_changes_spont[i][4] == 1]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by a decrease.')
print(repr(len([i for i in range(len(freq_expected_changes_spont)) if not np.isnan(freq_expected_changes_spont[i][0]) and freq_second_chosen_spont[i]]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were preceded by another, "smaller" increase.')
print(repr(len([i for i in range(len(freq_expected_changes_spont)) if not np.isnan(freq_expected_changes_spont[i][0]) and not np.isnan(freq_second_expected_changes_spont[i][0]) and not freq_second_chosen_spont[i]]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases were followed by another, "smaller" increase.')
print('Some of these groups may overlap.')
print('In ' + repr(len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[3])]) / len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])]) * 100) + ' % of these increases the signal was observed to return towards the baseline.')
print(repr(len([i for i in range(len(freq_expected_changes_spont)) if np.isnan(freq_expected_changes_spont[i][0]) and not np.isnan(freq_unexpected_changes_spont[i][0])]) / len(freq_expected_changes_spont) * 100) + ' % of the arousals showed an unaccompanied negative change (or two).')

print('\n' + repr(len([i for i in range(len(amp_eventual_expected_changes_spont)) if not np.isnan(amp_eventual_expected_changes_spont[i][0]) or not np.isnan(freq_eventual_expected_changes_spont[i][0])]) / len(amp_eventual_expected_changes_spont) * 100) + ' % of the arousals showed at least one type of expected change.')

In [ ]:
print('\ALL AROUSALS')
print('\nAmplitude:')
print(repr((len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])])) / (len(amp_eventual_expected_changes) + len(amp_eventual_expected_changes_spont)) * 100) + ' % of the arousals showed a negative change in amplitude (as expected).')
print('In ' + repr((len([change for change in amp_eventual_expected_changes if not np.isnan(change[3])]) + len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[3])])) / (len([change for change in amp_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in amp_eventual_expected_changes_spont if not np.isnan(change[0])])) * 100) + ' % of these decreases the signal was observed to return towards the baseline.')
print('\nFrequency:')
print(repr((len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])])) / (len(freq_eventual_expected_changes) + len(freq_eventual_expected_changes_spont)) * 100) + ' % of the arousals showed a positive change in frequency (as expected).')
print('In ' + repr((len([change for change in freq_eventual_expected_changes if not np.isnan(change[3])]) + len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[3])])) / (len([change for change in freq_eventual_expected_changes if not np.isnan(change[0])]) + len([change for change in freq_eventual_expected_changes_spont if not np.isnan(change[0])])) * 100) + ' % of these increases the signal was observed to return towards the baseline.')

print('\n' + repr((len([i for i in range(len(amp_eventual_expected_changes)) if not np.isnan(amp_eventual_expected_changes[i][0]) or not np.isnan(freq_eventual_expected_changes[i][0])]) + len([i for i in range(len(amp_eventual_expected_changes_spont)) if not np.isnan(amp_eventual_expected_changes_spont[i][0]) or not np.isnan(freq_eventual_expected_changes_spont[i][0])])) / (len(amp_eventual_expected_changes) + len(amp_eventual_expected_changes_spont)) * 100) + ' % of the arousals showed at least one type of expected change.')

## Demographics etc.

In [ ]:
print('Male:')
m = len([patient for patient in patients if demographics_by_patient[patient][1] == 1])
print(m)
print(repr(m/len(patients)) + ' %')
f = len([patient for patient in patients if demographics_by_patient[patient][1] == 0])
print(f)
print(repr(f/len(patients)) + ' %')

In [ ]:
print('Age:')
print(np.percentile([demographics_by_patient[patient][2] for patient in patients], [25, 50, 75]))
print('BMI:')
print(np.percentile([demographics_by_patient[patient][3] for patient in patients], [25, 50, 75]))

In [ ]:
tst = []
waso = []
ahi = []
ari = []
arri = []
stage_r = []
nrem = []
stage_n1 = []
stage_n2 = []
stage_n3 = []

for patient in patients:
    tst.append(len([stage for stage in stages[patient] if stage != 0])*30/3600)
    waso.append((len([stage for stage in stages[patient] if stage == 0])-np.nonzero(stages[patient])[0][0])*30/3600)
    ahi.append((len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/tst[-1])
    ari.append(len(arousal_inds_all[patient])/tst[-1])
    arri.append(len(arousal_inds_resp_all[patient])/tst[-1])
    stage_r.append(len([stage for stage in stages[patient] if stage == -1])/len([stage for stage in stages[patient] if stage != 0])*100)
    nrem.append(len([stage for stage in stages[patient] if stage < -1])/len([stage for stage in stages[patient] if stage != 0])*100)
    stage_n1.append(len([stage for stage in stages[patient] if stage == -2])/len([stage for stage in stages[patient] if stage != 0])*100)
    stage_n2.append(len([stage for stage in stages[patient] if stage == -3])/len([stage for stage in stages[patient] if stage != 0])*100)
    stage_n3.append(len([stage for stage in stages[patient] if stage == -4])/len([stage for stage in stages[patient] if stage != 0])*100)

print('TST:')
print(np.percentile(tst, [25, 50, 75]))
print('WASO:')
print(np.percentile(waso, [25, 50, 75]))
print('AHI:')
print(np.percentile(ahi, [25, 50, 75]))
print('ArI:')
print(np.percentile(ari, [25, 50, 75]))
print('ArRI:')
print(np.percentile(arri, [25, 50, 75]))
print('Stage R:')
print(np.percentile(stage_r, [25, 50, 75]))
print('NREM:')
print(np.percentile(nrem, [25, 50, 75]))
print('Stage N1:')
print(np.percentile(stage_n1, [25, 50, 75]))
print('Stage N2:')
print(np.percentile(stage_n2, [25, 50, 75]))
print('Stage N3:')
print(np.percentile(stage_n3, [25, 50, 75]))

## Effect of sleep stage

In [ ]:
# indices corresponding to lists produced by the models, in four groups: R, N1, N2, N3
indices_by_stage = [[i for i in range(len(labels)) if stages2[labels[i][0]][arousal_inds[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] == stage] for stage in [-1, -2, -3, -4]]
indices_by_stage_spont = [[i for i in range(len(labels_spont)) if stages2[labels_spont[i][0]][arousal_inds_spont[labels_spont[i][0]][arousals_with_10s10s_margins_spont[labels_spont[i][0]][labels_spont[i][1]]]] == stage] for stage in [-1, -2, -3, -4]]

In [ ]:
print('Stage R:')
print(len(indices_by_stage[0] + indices_by_stage_spont[0]))
print((len(indices_by_stage[0] + indices_by_stage_spont[0]))/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('NREM:')
print(len(indices_by_stage[1] + indices_by_stage_spont[1] + indices_by_stage[2] + indices_by_stage_spont[2] + indices_by_stage[3] + indices_by_stage_spont[3]))
print(len(indices_by_stage[1] + indices_by_stage_spont[1] + indices_by_stage[2] + indices_by_stage_spont[2] + indices_by_stage[3] + indices_by_stage_spont[3])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Stage N1:')
print(len(indices_by_stage[1] + indices_by_stage_spont[1]))
print(len(indices_by_stage[1] + indices_by_stage_spont[1])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Stage N2:')
print(len(indices_by_stage[2] + indices_by_stage_spont[2]))
print(len(indices_by_stage[2] + indices_by_stage_spont[2])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Stage N3:')
print(len(indices_by_stage[3] + indices_by_stage_spont[3]))
print(len(indices_by_stage[3] + indices_by_stage_spont[3])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))

In [ ]:
freq_magnitudes_by_stage = [[freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_stage[i] if not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_stage_spont[i] if not np.isnan(freq_startbaselines_spont[j])] for i in range(4)]
amp_magnitudes_by_stage = [[-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_stage[i] if not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_stage_spont[i] if not np.isnan(amp_startbaselines_spont[j])] for i in range(4)]

In [ ]:
freq_magnitudes_by_stage_patients = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(freq_startbaselines[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(freq_startbaselines_spont[j])] for i in range(4)]
amp_magnitudes_by_stage_patients = [[labels[j][0] for j in indices_by_stage[i] if not np.isnan(amp_startbaselines[j])] + [labels_spont[j][0] for j in indices_by_stage_spont[i] if not np.isnan(amp_startbaselines_spont[j])] for i in range(4)]

In [ ]:
# This takes a long time. At the end it saves the results, so you shouldn't have to run this many times.

freq_dir_mat = np.zeros([4,4])
freq_pval_mat = []
freq_pval_precise_mat = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    if np.median(freq_magnitudes_by_stage[i]) < np.median(freq_magnitudes_by_stage[j]):
        alternative = 'less'
        freq_dir_mat[i, j] = -1
    else:
        alternative = 'greater'
        freq_dir_mat[i, j] = 1
    pvalue = iterative_wilcoxon(freq_magnitudes_by_stage[i], freq_magnitudes_by_stage[j], freq_magnitudes_by_stage_patients[i], freq_magnitudes_by_stage_patients[j], alternative, 1000)[0]
    freq_pval_precise_mat.append(((i, j), pvalue))
    if pvalue < 0.001:
        freq_pval_mat.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        freq_pval_mat.append(((i, j), '%.3f' % pvalue))
    else:
        freq_pval_mat.append(((i, j), '%.2f' % pvalue))
with open('/wrk/luukinen/results/20240221/freqstagemats.pickle', 'wb') as f:
    pickle.dump((freq_dir_mat, freq_pval_mat, freq_pval_precise_mat), f)

In [ ]:
# Same thing as above applies here too.

amp_dir_mat = np.zeros([4,4])
amp_pval_mat = []
amp_pval_precise_mat = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    if np.median(amp_magnitudes_by_stage[i]) < np.median(amp_magnitudes_by_stage[j]):
        alternative = 'less'
        amp_dir_mat[i, j] = -1
    else:
        alternative = 'greater'
        amp_dir_mat[i, j] = 1
    pvalue = iterative_wilcoxon(amp_magnitudes_by_stage[i], amp_magnitudes_by_stage[j], amp_magnitudes_by_stage_patients[i], amp_magnitudes_by_stage_patients[j], alternative, 1000)[0]
    amp_pval_precise_mat.append(((i, j), pvalue))
    if pvalue < 0.001:
        amp_pval_mat.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        amp_pval_mat.append(((i, j), '%.3f' % pvalue))
    else:
        amp_pval_mat.append(((i, j), '%.2f' % pvalue))
with open('/wrk/luukinen/results/20240221/ampstagemats.pickle', 'wb') as f:
    pickle.dump((amp_dir_mat, amp_pval_mat, amp_pval_precise_mat), f)

In [ ]:
# Very slow.

p_freq_magnitude_N1N2 = iterative_wilcoxon(freq_magnitudes_by_stage[1], freq_magnitudes_by_stage[2], freq_magnitudes_by_stage_patients[1], freq_magnitudes_by_stage_patients[2], 'less', 1000)
p_freq_magnitude_N2N3 = iterative_wilcoxon(freq_magnitudes_by_stage[2], freq_magnitudes_by_stage[3], freq_magnitudes_by_stage_patients[2], freq_magnitudes_by_stage_patients[3], 'less', 1000)
print(p_freq_magnitude_N1N2[0])
print(p_freq_magnitude_N2N3[0])

In [ ]:
# Very slow.

p_amp_magnitude_RN1 = iterative_wilcoxon(amp_magnitudes_by_stage[0], amp_magnitudes_by_stage[1], amp_magnitudes_by_stage_patients[0], amp_magnitudes_by_stage_patients[1], 'less', 1000)
p_amp_magnitude_N1N2 = iterative_wilcoxon(amp_magnitudes_by_stage[1], amp_magnitudes_by_stage[2], amp_magnitudes_by_stage_patients[1], amp_magnitudes_by_stage_patients[2], 'less', 1000)
p_amp_magnitude_N2N3 = iterative_wilcoxon(amp_magnitudes_by_stage[2], amp_magnitudes_by_stage[3], amp_magnitudes_by_stage_patients[2], amp_magnitudes_by_stage_patients[3], 'less', 1000)
print(p_amp_magnitude_RN1[0])
print(p_amp_magnitude_N1N2[0])
print(p_amp_magnitude_N2N3[0])

In [ ]:
[list(np.unique(l)) for l in amp_magnitudes_by_stage_patients] == [list(np.unique(l)) for l in freq_magnitudes_by_stage_patients]

In [ ]:
patients_with_arousals_in_all_stages = [patient for patient in patients if np.all([patient in l for l in freq_magnitudes_by_stage_patients])]
patients_with_arousals_in_all_NREM_stages = [patient for patient in patients if patient in freq_magnitudes_by_stage_patients[1] and patient in freq_magnitudes_by_stage_patients[2] and patient in freq_magnitudes_by_stage_patients[3]]
patients_with_arousals_in_N1_and_N2 = [patient for patient in patients if patient in freq_magnitudes_by_stage_patients[1] and patient in freq_magnitudes_by_stage_patients[2]]

### Normalization

In [ ]:
# indices corresponding to lists produced by the models, in three groups: apneas with desat, apneas without desat, hypopneas with desat and hypopneas without desat. The desaturation threshold is 3 % (inclusive)
arousals_by_type = [
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3],
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3],
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3],
    [i for i in range(len(labels)) if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3],
]
[len(l) for l in arousals_by_type]

In [ ]:
print('Apnea, Des:')
print(len(arousals_by_type[0])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Apnea, no Des:')
print(len(arousals_by_type[1])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Hypopnea, Des:')
print(len(arousals_by_type[2])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Hypopnea, no Des:')
print(len(arousals_by_type[3])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))
print('Spontaneous:')
print(sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])/(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]) + sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients])))

In [ ]:
freq_changes_by_arousal_type1 = [[freq_eventual_expected_changes[i][0] - freq_startbaselines[i] if not np.isnan(freq_eventual_expected_changes[i][0]) else 0 for i in l if not np.isnan(freq_startbaselines[i])] for l in arousals_by_type] + [[freq_eventual_expected_changes_spont[i][0] - freq_startbaselines_spont[i] if not np.isnan(freq_eventual_expected_changes_spont[i][0]) else 0 for i in range(len(freq_eventual_expected_changes_spont)) if not np.isnan(freq_startbaselines_spont[i])]]
amp_changes_by_arousal_type1 = [[-amp_eventual_expected_changes[i][0] + amp_startbaselines[i] if not np.isnan(amp_eventual_expected_changes[i][0]) else 0 for i in l if not np.isnan(amp_startbaselines[i])] for l in arousals_by_type] + [[-amp_eventual_expected_changes_spont[i][0] + amp_startbaselines_spont[i] if not np.isnan(amp_eventual_expected_changes_spont[i][0]) else 0 for i in range(len(amp_eventual_expected_changes_spont)) if not np.isnan(amp_startbaselines_spont[i])]]

In [ ]:
median_freq_magnitude_by_type = [np.median(l) for l in freq_changes_by_arousal_type1]
median_amp_magnitude_by_type = [np.median(l) for l in amp_changes_by_arousal_type1]

In [ ]:
arousal_types = []
for i in range(len(labels)):
    if related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3:
        arousal_types.append(0)
    elif related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in apnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3:
        arousal_types.append(1)
    elif related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] >= 3:
        arousal_types.append(2)
    elif related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]] in hypopnea_inds[labels[i][0]] and desats[labels[i][0]][related_events[labels[i][0]][arousals_with_10s10s_margins[labels[i][0]][labels[i][1]]]] < 3:
        arousal_types.append(3)
    else:
        raise Exception('Related event uncategorized! This should not happen. You have probably broken something.')

In [ ]:
freq_magnitudes_by_stage_normalized = [[(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_stage[i] if not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_stage_spont[i] if not np.isnan(freq_startbaselines_spont[j])] for i in range(4)]
amp_magnitudes_by_stage_normalized = [[(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_stage[i] if not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_stage_spont[i] if not np.isnan(amp_startbaselines_spont[j])] for i in range(4)]

In [ ]:
# Again, very slow but saves its results.

freq_dir_mat_normalized = np.zeros([4,4])
freq_pval_mat_normalized = []
freq_pval_precise_mat_normalized = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    if np.median(freq_magnitudes_by_stage_normalized[i]) < np.median(freq_magnitudes_by_stage_normalized[j]):
        alternative = 'less'
        freq_dir_mat_normalized[i, j] = -1
    else:
        alternative = 'greater'
        freq_dir_mat_normalized[i, j] = 1
    pvalue = iterative_wilcoxon(freq_magnitudes_by_stage_normalized[i], freq_magnitudes_by_stage_normalized[j], freq_magnitudes_by_stage_patients[i], freq_magnitudes_by_stage_patients[j], alternative, 1000)[0]
    freq_pval_precise_mat_normalized.append(((i, j), pvalue))
    if pvalue < 0.001:
        freq_pval_mat_normalized.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        freq_pval_mat_normalized.append(((i, j), '%.3f' % pvalue))
    else:
        freq_pval_mat_normalized.append(((i, j), '%.2f' % pvalue))
with open('/wrk/luukinen/results/20240221/freqstagematsnormalized.pickle', 'wb') as f:
    pickle.dump((freq_dir_mat_normalized, freq_pval_mat_normalized, freq_pval_precise_mat_normalized), f)

In [ ]:
# Ditto.

amp_dir_mat_normalized = np.zeros([4,4])
amp_pval_mat_normalized = []
amp_pval_precise_mat_normalized = []

for i, j in tqdm(itertools.combinations([0,1,2,3], 2)):
    if np.median(amp_magnitudes_by_stage_normalized[i]) < np.median(amp_magnitudes_by_stage_normalized[j]):
        alternative = 'less'
        amp_dir_mat_normalized[i, j] = -1
    else:
        alternative = 'greater'
        amp_dir_mat_normalized[i, j] = 1
    pvalue = iterative_wilcoxon(amp_magnitudes_by_stage_normalized[i], amp_magnitudes_by_stage_normalized[j], amp_magnitudes_by_stage_patients[i], amp_magnitudes_by_stage_patients[j], alternative, 1000)[0]
    amp_pval_precise_mat_normalized.append(((i, j), pvalue))
    if pvalue < 0.001:
        amp_pval_mat_normalized.append(((i, j), '<1e-3'))
    elif pvalue < 0.01:
        amp_pval_mat_normalized.append(((i, j), '%.3f' % pvalue))
    else:
        amp_pval_mat_normalized.append(((i, j), '%.2f' % pvalue))
with open('/wrk/luukinen/results/20240221/ampstagematsnormalized.pickle', 'wb') as f:
    pickle.dump((amp_dir_mat_normalized, amp_pval_mat_normalized, amp_pval_precise_mat_normalized), f)

### Figures

In [ ]:
if 9690 in plt.get_fignums(): plt.close(9690)
fig= plt.figure(num=9690, figsize=(17/2.54, 6), dpi=1000)
gs = GridSpec(2, 2, width_ratios=[2,1])
ax = []
ax.append(fig.add_subplot(gs[0, 0]))
ax.append(fig.add_subplot(gs[1, 0]))
ax[0].violinplot(freq_magnitudes_by_stage, showextrema=False, showmedians=True)
ax[0].set_ylim(0, np.percentile([n for l in freq_magnitudes_by_stage for n in l], 90))
ax[1].violinplot(freq_magnitudes_by_stage_normalized, showextrema=False, showmedians=True)
ax[1].set_ylim(0, np.percentile([n for l in freq_magnitudes_by_stage_normalized for n in l], 90))
for i in [0,1]:
    ax[i].set_xticks([1,2,3,4])
    plt.setp(ax[i].get_yticklabels(), **figfont10)
    ax[i].set_xticklabels(['R', 'N1', 'N2', 'N3'], **figfont10)
ax[0].set_ylabel('Frequency increase (Hz)', **figfont10)
ax[1].set_ylabel('Normalized frequency increase (a.u.)', **figfont10)
ax[0].set_xlabel('(a)', **figfont8, labelpad=8)
ax[1].set_xlabel('(c)', **figfont8, labelpad=8)

plt.tight_layout()
plt.show()

with open('/wrk/luukinen/results/20240221/freqstagemats.pickle', 'rb') as f:
    freq_dir_mat, freq_pval_mat, freq_pval_precise_mat = pickle.load(f)
with open('/wrk/luukinen/results/20240221/freqstagematsnormalized.pickle', 'rb') as f:
    freq_dir_mat_normalized, freq_pval_mat_normalized, freq_pval_precise_mat_normalized = pickle.load(f)

indlabels = ['R', 'N1', 'N2', 'N3']

scale = 2
colornorm = colors.TwoSlopeNorm(vmin=-scale, vcenter=0, vmax=scale)
ax.append(fig.add_subplot(gs[0, 1]))
ax.append(fig.add_subplot(gs[1, 1]))
ax[2].matshow(freq_dir_mat, norm=colornorm, cmap='bwr')
ax[2].tick_params(axis="x", bottom=False, top=True, labelbottom=False, labeltop=True)
ax[2].tick_params(axis="y", left=False, right=True, labelleft=False, labelright=True)
# ax[2].set_frame_on(False)
ax[2].spines['left'].set_visible(False)
ax[2].spines['bottom'].set_visible(False)
ax[2].set_xticklabels(['']+indlabels)
ax[2].set_yticklabels(['']+indlabels)
# plt.setp(ax[2].get_xticklabels(), **figfont10, ha='left', va='bottom')
# plt.setp(ax[2].get_yticklabels(), **figfont10, ha='left', va='bottom')
plt.setp(ax[2].get_xticklabels(), **figfont10, ha='center', va='center')
plt.setp(ax[2].get_yticklabels(), **figfont10, ha='left', va='center')
# ax[2].tick_params(axis='both', labelrotation=45)
ax[2].set_xlabel('(b)', **figfont8, labelpad=8)
# ax[2].xaxis.get_label().set_y(ax[0].xaxis.get_label().get_position()[1]-100)
for (j, i), text in freq_pval_mat:
    ax[2].text(i, j, text, va='center', ha='center', **figfont8)
for i in range(4):
    ax[2].text(i, i, '-', va='center', ha='center', **figfont8)
ax[3].matshow(freq_dir_mat_normalized, norm=colornorm, cmap='bwr')
ax[3].tick_params(axis="x", bottom=False, top=True, labelbottom=False, labeltop=True)
ax[3].tick_params(axis="y", left=False, right=True, labelleft=False, labelright=True)
# ax[3].set_frame_on(False)
ax[3].spines['left'].set_visible(False)
ax[3].spines['bottom'].set_visible(False)
ax[3].set_xticklabels(['']*6)
ax[3].set_yticklabels(['']+indlabels)
# plt.setp(ax[3].get_xticklabels(), **figfont10, ha='left', va='bottom')
# plt.setp(ax[3].get_yticklabels(), **figfont10, ha='left', va='bottom')
plt.setp(ax[3].get_xticklabels(), **figfont10, ha='center', va='center')
plt.setp(ax[3].get_yticklabels(), **figfont10, ha='left', va='center')
# ax[3].tick_params(axis='both', labelrotation=45)
ax[3].set_xlabel('(d)', **figfont8, labelpad=8)
for (j, i), text in freq_pval_mat_normalized:
    ax[3].text(i, j, text, va='center', ha='center', **figfont8)
for i in range(4):
    ax[3].text(i, i, '-', va='center', ha='center', **figfont8)
fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('fig1.png')

In [ ]:
print([np.median(l) for l in freq_magnitudes_by_stage])
print([np.median(l) for l in freq_magnitudes_by_stage_normalized])

In [ ]:
if 8138 in plt.get_fignums(): plt.close(8138)
fig= plt.figure(num=8138, figsize=(17/2.54, 6), dpi=1000)
gs = GridSpec(2, 2, width_ratios=[2,1])
ax = []
ax.append(fig.add_subplot(gs[0, 0]))
ax.append(fig.add_subplot(gs[1, 0]))
ax[0].violinplot(amp_magnitudes_by_stage, showextrema=False, showmedians=True)
ax[0].set_ylim(0, np.percentile([n for l in amp_magnitudes_by_stage for n in l], 90))
ax[1].violinplot(amp_magnitudes_by_stage_normalized, showextrema=False, showmedians=True)
ax[1].set_ylim(0, np.percentile([n for l in amp_magnitudes_by_stage_normalized for n in l], 90))
for i in [0,1]:
    ax[i].set_xticks([1,2,3,4])
    plt.setp(ax[i].get_yticklabels(), **figfont10)
    ax[i].set_xticklabels(['R', 'N1', 'N2', 'N3'], **figfont10)
ax[0].set_ylabel('Amplitude decrease (a.u.)', **figfont10)
ax[1].set_ylabel('Normalized amplitude decrease (a.u.)', **figfont10)
ax[0].set_xlabel('(a)', **figfont8, labelpad=8)
ax[1].set_xlabel('(c)', **figfont8, labelpad=8)

plt.tight_layout()
plt.show()

with open('/wrk/luukinen/results/20240221/ampstagemats.pickle', 'rb') as f:
    amp_dir_mat, amp_pval_mat, amp_pval_precise_mat = pickle.load(f)
with open('/wrk/luukinen/results/20240221/ampstagematsnormalized.pickle', 'rb') as f:
    amp_dir_mat_normalized, amp_pval_mat_normalized, amp_pval_precise_mat_normalized = pickle.load(f)

indlabels = ['R', 'N1', 'N2', 'N3']

scale = 2
colornorm = colors.TwoSlopeNorm(vmin=-scale, vcenter=0, vmax=scale)
ax.append(fig.add_subplot(gs[0, 1]))
ax.append(fig.add_subplot(gs[1, 1]))
ax[2].matshow(amp_dir_mat, norm=colornorm, cmap='bwr')
ax[2].tick_params(axis="x", bottom=False, top=True, labelbottom=False, labeltop=True)
ax[2].tick_params(axis="y", left=False, right=True, labelleft=False, labelright=True)
# ax[2].set_frame_on(False)
ax[2].spines['left'].set_visible(False)
ax[2].spines['bottom'].set_visible(False)
ax[2].set_xticklabels(['']+indlabels)
ax[2].set_yticklabels(['']+indlabels)
# plt.setp(ax[2].get_xticklabels(), **figfont10, ha='left', va='bottom')
# plt.setp(ax[2].get_yticklabels(), **figfont10, ha='left', va='bottom')
plt.setp(ax[2].get_xticklabels(), **figfont10, ha='center', va='center')
plt.setp(ax[2].get_yticklabels(), **figfont10, ha='left', va='center')
# ax[2].tick_params(axis='both', labelrotation=45)
ax[2].set_xlabel('(b)', **figfont8, labelpad=8)
# ax[2].xaxis.get_label().set_y(ax[0].xaxis.get_label().get_position()[1]-100)
for (j, i), text in amp_pval_mat:
    ax[2].text(i, j, text, va='center', ha='center', **figfont8)
for i in range(4):
    ax[2].text(i, i, '-', va='center', ha='center', **figfont8)
ax[3].matshow(amp_dir_mat_normalized, norm=colornorm, cmap='bwr')
ax[3].tick_params(axis="x", bottom=False, top=True, labelbottom=False, labeltop=True)
ax[3].tick_params(axis="y", left=False, right=True, labelleft=False, labelright=True)
# ax[3].set_frame_on(False)
ax[3].spines['left'].set_visible(False)
ax[3].spines['bottom'].set_visible(False)
ax[3].set_xticklabels(['']*6)
ax[3].set_yticklabels(['']+indlabels)
# plt.setp(ax[3].get_xticklabels(), **figfont10, ha='left', va='bottom')
# plt.setp(ax[3].get_yticklabels(), **figfont10, ha='left', va='bottom')
plt.setp(ax[3].get_xticklabels(), **figfont10, ha='center', va='center')
plt.setp(ax[3].get_yticklabels(), **figfont10, ha='left', va='center')
# ax[3].tick_params(axis='both', labelrotation=45)
ax[3].set_xlabel('(d)', **figfont8, labelpad=8)
for (j, i), text in amp_pval_mat_normalized:
    ax[3].text(i, j, text, va='center', ha='center', **figfont8)
for i in range(4):
    ax[3].text(i, i, '-', va='center', ha='center', **figfont8)
fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('fig2.png')

In [ ]:
print([np.median(l) for l in amp_magnitudes_by_stage])
print([np.median(l) for l in amp_magnitudes_by_stage_normalized])

## Patientwise variation

In [ ]:
indices_by_patient = [[i for i in range(len(labels)) if labels[i][0] == patient] for patient in tqdm(patients)]
indices_by_patient_spont = [[i for i in range(len(labels_spont)) if labels_spont[i][0] == patient] for patient in tqdm(patients)]

In [ ]:
freq_magnitudes_by_patient = [[freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient = [[-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]

In [ ]:
print('ALL STAGES\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - ALL STAGES')
q1_freq_magnitude_interquartile_range_all_stages, q3_freq_magnitude_interquartile_range_all_stages = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_all_stages = [patients[i] for i in range(len(patients)) if np.quantile(freq_magnitudes_by_patient[i], 0.75) - np.quantile(freq_magnitudes_by_patient[i], 0.25) < q1_freq_magnitude_interquartile_range_all_stages]
patients_in_q4_freq_magnitude_interquartile_range_all_stages = [patients[i] for i in range(len(patients)) if np.quantile(freq_magnitudes_by_patient[i], 0.75) - np.quantile(freq_magnitudes_by_patient[i], 0.25) > q3_freq_magnitude_interquartile_range_all_stages]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_all_stages = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages]
ahi_q4_freq_magnitude_interquartile_range_all_stages = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_all_stages))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_all_stages, ahi_q4_freq_magnitude_interquartile_range_all_stages, alternative='less'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_all_stages = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages]
dessevs_q4_freq_magnitude_interquartile_range_all_stages = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_all_stages))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_all_stages, dessevs_q4_freq_magnitude_interquartile_range_all_stages, alternative='less'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_all_stages = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages]
lapses_q4_freq_magnitude_interquartile_range_all_stages = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_all_stages))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_all_stages, lapses_q4_freq_magnitude_interquartile_range_all_stages, alternative='less'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_all_stages = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages]
speed_q4_freq_magnitude_interquartile_range_all_stages = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages]
print(np.median(speed_q1_freq_magnitude_interquartile_range_all_stages))
print(np.median(speed_q4_freq_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_all_stages, speed_q4_freq_magnitude_interquartile_range_all_stages, alternative='greater'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - ALL STAGES')
q1_amp_magnitude_interquartile_range_all_stages, q3_amp_magnitude_interquartile_range_all_stages = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_all_stages = [patients[i] for i in range(len(patients)) if np.quantile(amp_magnitudes_by_patient[i], 0.75) - np.quantile(amp_magnitudes_by_patient[i], 0.25) < q1_amp_magnitude_interquartile_range_all_stages]
patients_in_q4_amp_magnitude_interquartile_range_all_stages = [patients[i] for i in range(len(patients)) if np.quantile(amp_magnitudes_by_patient[i], 0.75) - np.quantile(amp_magnitudes_by_patient[i], 0.25) > q3_amp_magnitude_interquartile_range_all_stages]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_all_stages = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages]
ahi_q4_amp_magnitude_interquartile_range_all_stages = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_all_stages))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_all_stages, ahi_q4_amp_magnitude_interquartile_range_all_stages, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_all_stages = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages]
dessevs_q4_amp_magnitude_interquartile_range_all_stages = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_all_stages))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_all_stages, dessevs_q4_amp_magnitude_interquartile_range_all_stages, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_all_stages = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages]
lapses_q4_amp_magnitude_interquartile_range_all_stages = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_all_stages))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_all_stages, lapses_q4_amp_magnitude_interquartile_range_all_stages, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_all_stages = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages]
speed_q4_amp_magnitude_interquartile_range_all_stages = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages]
print(np.median(speed_q1_amp_magnitude_interquartile_range_all_stages))
print(np.median(speed_q4_amp_magnitude_interquartile_range_all_stages))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_all_stages, speed_q4_amp_magnitude_interquartile_range_all_stages, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_R = [[freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_R = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_R[i]) > 1]
freq_magnitudes_by_patient_R = [l for l in freq_magnitudes_by_patient_R if len(l) > 1]

amp_magnitudes_by_patient_R = [[-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_R = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_R[i]) > 1]
amp_magnitudes_by_patient_R = [l for l in amp_magnitudes_by_patient_R if len(l) > 1]

In [ ]:
print('STAGE R\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE R')
q1_freq_magnitude_interquartile_range_R, q3_freq_magnitude_interquartile_range_R = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_R = [patients_freq_magnitudes_R[i] for i in range(len(patients_freq_magnitudes_R)) if np.quantile(freq_magnitudes_by_patient_R[i], 0.75) - np.quantile(freq_magnitudes_by_patient_R[i], 0.25) < q1_freq_magnitude_interquartile_range_R]
patients_in_q4_freq_magnitude_interquartile_range_R = [patients_freq_magnitudes_R[i] for i in range(len(patients_freq_magnitudes_R)) if np.quantile(freq_magnitudes_by_patient_R[i], 0.75) - np.quantile(freq_magnitudes_by_patient_R[i], 0.25) > q3_freq_magnitude_interquartile_range_R]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_R = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_R]
ahi_q4_freq_magnitude_interquartile_range_R = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_R]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_R))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_R, ahi_q4_freq_magnitude_interquartile_range_R, alternative='greater'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_R = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_R]
dessevs_q4_freq_magnitude_interquartile_range_R = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_R]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_R))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_R, dessevs_q4_freq_magnitude_interquartile_range_R, alternative='greater'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_R = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_R]
lapses_q4_freq_magnitude_interquartile_range_R = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_R]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_R))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_R, lapses_q4_freq_magnitude_interquartile_range_R, alternative='greater'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_R = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_R]
speed_q4_freq_magnitude_interquartile_range_R = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_R]
print(np.median(speed_q1_freq_magnitude_interquartile_range_R))
print(np.median(speed_q4_freq_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_R, speed_q4_freq_magnitude_interquartile_range_R, alternative='less'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE R')
q1_amp_magnitude_interquartile_range_R, q3_amp_magnitude_interquartile_range_R = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_R = [patients_amp_magnitudes_R[i] for i in range(len(patients_amp_magnitudes_R)) if np.quantile(amp_magnitudes_by_patient_R[i], 0.75) - np.quantile(amp_magnitudes_by_patient_R[i], 0.25) < q1_amp_magnitude_interquartile_range_R]
patients_in_q4_amp_magnitude_interquartile_range_R = [patients_amp_magnitudes_R[i] for i in range(len(patients_amp_magnitudes_R)) if np.quantile(amp_magnitudes_by_patient_R[i], 0.75) - np.quantile(amp_magnitudes_by_patient_R[i], 0.25) > q3_amp_magnitude_interquartile_range_R]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_R = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_R]
ahi_q4_amp_magnitude_interquartile_range_R = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_R]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_R))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_R, ahi_q4_amp_magnitude_interquartile_range_R, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_R = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_R]
dessevs_q4_amp_magnitude_interquartile_range_R = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_R]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_R))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_R, dessevs_q4_amp_magnitude_interquartile_range_R, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_R = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_R]
lapses_q4_amp_magnitude_interquartile_range_R = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_R]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_R))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_R, lapses_q4_amp_magnitude_interquartile_range_R, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_R = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_R]
speed_q4_amp_magnitude_interquartile_range_R = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_R]
print(np.median(speed_q1_amp_magnitude_interquartile_range_R))
print(np.median(speed_q4_amp_magnitude_interquartile_range_R))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_R, speed_q4_amp_magnitude_interquartile_range_R, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_N1 = [[freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_N1 = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_N1[i]) > 1]
freq_magnitudes_by_patient_N1 = [l for l in freq_magnitudes_by_patient_N1 if len(l) > 1]

amp_magnitudes_by_patient_N1 = [[-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_N1 = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_N1[i]) > 1]
amp_magnitudes_by_patient_N1 = [l for l in amp_magnitudes_by_patient_N1 if len(l) > 1]

In [ ]:
print('STAGE N1\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE N1')
q1_freq_magnitude_interquartile_range_N1, q3_freq_magnitude_interquartile_range_N1 = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_N1 = [patients_freq_magnitudes_N1[i] for i in range(len(patients_freq_magnitudes_N1)) if np.quantile(freq_magnitudes_by_patient_N1[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N1[i], 0.25) < q1_freq_magnitude_interquartile_range_N1]
patients_in_q4_freq_magnitude_interquartile_range_N1 = [patients_freq_magnitudes_N1[i] for i in range(len(patients_freq_magnitudes_N1)) if np.quantile(freq_magnitudes_by_patient_N1[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N1[i], 0.25) > q3_freq_magnitude_interquartile_range_N1]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_N1 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_N1]
ahi_q4_freq_magnitude_interquartile_range_N1 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_N1]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_N1))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_N1, ahi_q4_freq_magnitude_interquartile_range_N1, alternative='less'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_N1 = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_N1]
dessevs_q4_freq_magnitude_interquartile_range_N1 = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_N1]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_N1))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_N1, dessevs_q4_freq_magnitude_interquartile_range_N1, alternative='less'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_N1 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N1]
lapses_q4_freq_magnitude_interquartile_range_N1 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N1]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_N1))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_N1, lapses_q4_freq_magnitude_interquartile_range_N1, alternative='greater'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_N1 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N1]
speed_q4_freq_magnitude_interquartile_range_N1 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N1]
print(np.median(speed_q1_freq_magnitude_interquartile_range_N1))
print(np.median(speed_q4_freq_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_N1, speed_q4_freq_magnitude_interquartile_range_N1, alternative='greater'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE N1')
q1_amp_magnitude_interquartile_range_N1, q3_amp_magnitude_interquartile_range_N1 = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_N1 = [patients_amp_magnitudes_N1[i] for i in range(len(patients_amp_magnitudes_N1)) if np.quantile(amp_magnitudes_by_patient_N1[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N1[i], 0.25) < q1_amp_magnitude_interquartile_range_N1]
patients_in_q4_amp_magnitude_interquartile_range_N1 = [patients_amp_magnitudes_N1[i] for i in range(len(patients_amp_magnitudes_N1)) if np.quantile(amp_magnitudes_by_patient_N1[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N1[i], 0.25) > q3_amp_magnitude_interquartile_range_N1]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_N1 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_N1]
ahi_q4_amp_magnitude_interquartile_range_N1 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_N1]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_N1))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_N1, ahi_q4_amp_magnitude_interquartile_range_N1, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_N1 = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_N1]
dessevs_q4_amp_magnitude_interquartile_range_N1 = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_N1]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_N1))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_N1, dessevs_q4_amp_magnitude_interquartile_range_N1, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_N1 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N1]
lapses_q4_amp_magnitude_interquartile_range_N1 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N1]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_N1))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_N1, lapses_q4_amp_magnitude_interquartile_range_N1, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_N1 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N1]
speed_q4_amp_magnitude_interquartile_range_N1 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N1]
print(np.median(speed_q1_amp_magnitude_interquartile_range_N1))
print(np.median(speed_q4_amp_magnitude_interquartile_range_N1))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_N1, speed_q4_amp_magnitude_interquartile_range_N1, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_N2 = [[freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_N2 = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_N2[i]) > 1]
freq_magnitudes_by_patient_N2 = [l for l in freq_magnitudes_by_patient_N2 if len(l) > 1]

amp_magnitudes_by_patient_N2 = [[-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_N2 = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_N2[i]) > 1]
amp_magnitudes_by_patient_N2 = [l for l in amp_magnitudes_by_patient_N2 if len(l) > 1]

In [ ]:
print('STAGE N2\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE N2')
q1_freq_magnitude_interquartile_range_N2, q3_freq_magnitude_interquartile_range_N2 = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_N2 = [patients_freq_magnitudes_N2[i] for i in range(len(patients_freq_magnitudes_N2)) if np.quantile(freq_magnitudes_by_patient_N2[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N2[i], 0.25) < q1_freq_magnitude_interquartile_range_N2]
patients_in_q4_freq_magnitude_interquartile_range_N2 = [patients_freq_magnitudes_N2[i] for i in range(len(patients_freq_magnitudes_N2)) if np.quantile(freq_magnitudes_by_patient_N2[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N2[i], 0.25) > q3_freq_magnitude_interquartile_range_N2]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_N2 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_N2]
ahi_q4_freq_magnitude_interquartile_range_N2 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_N2]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_N2))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_N2, ahi_q4_freq_magnitude_interquartile_range_N2, alternative='less'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_N2 = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_N2]
dessevs_q4_freq_magnitude_interquartile_range_N2 = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_N2]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_N2))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_N2, dessevs_q4_freq_magnitude_interquartile_range_N2, alternative='less'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_N2 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N2]
lapses_q4_freq_magnitude_interquartile_range_N2 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N2]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_N2))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_N2, lapses_q4_freq_magnitude_interquartile_range_N2, alternative='less'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_N2 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N2]
speed_q4_freq_magnitude_interquartile_range_N2 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N2]
print(np.median(speed_q1_freq_magnitude_interquartile_range_N2))
print(np.median(speed_q4_freq_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_N2, speed_q4_freq_magnitude_interquartile_range_N2, alternative='greater'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE N2')
q1_amp_magnitude_interquartile_range_N2, q3_amp_magnitude_interquartile_range_N2 = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_N2 = [patients_amp_magnitudes_N2[i] for i in range(len(patients_amp_magnitudes_N2)) if np.quantile(amp_magnitudes_by_patient_N2[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N2[i], 0.25) < q1_amp_magnitude_interquartile_range_N2]
patients_in_q4_amp_magnitude_interquartile_range_N2 = [patients_amp_magnitudes_N2[i] for i in range(len(patients_amp_magnitudes_N2)) if np.quantile(amp_magnitudes_by_patient_N2[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N2[i], 0.25) > q3_amp_magnitude_interquartile_range_N2]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_N2 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_N2]
ahi_q4_amp_magnitude_interquartile_range_N2 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_N2]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_N2))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_N2, ahi_q4_amp_magnitude_interquartile_range_N2, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_N2 = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_N2]
dessevs_q4_amp_magnitude_interquartile_range_N2 = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_N2]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_N2))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_N2, dessevs_q4_amp_magnitude_interquartile_range_N2, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_N2 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N2]
lapses_q4_amp_magnitude_interquartile_range_N2 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N2]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_N2))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_N2, lapses_q4_amp_magnitude_interquartile_range_N2, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_N2 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N2]
speed_q4_amp_magnitude_interquartile_range_N2 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N2]
print(np.median(speed_q1_amp_magnitude_interquartile_range_N2))
print(np.median(speed_q4_amp_magnitude_interquartile_range_N2))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_N2, speed_q4_amp_magnitude_interquartile_range_N2, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_N3 = [[freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_freq_magnitudes_N3 = [patients[i] for i in range(len(patients)) if len(freq_magnitudes_by_patient_N3[i]) > 1]
freq_magnitudes_by_patient_N3 = [l for l in freq_magnitudes_by_patient_N3 if len(l) > 1]

amp_magnitudes_by_patient_N3 = [[-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
patients_amp_magnitudes_N3 = [patients[i] for i in range(len(patients)) if len(amp_magnitudes_by_patient_N3[i]) > 1]
amp_magnitudes_by_patient_N3 = [l for l in amp_magnitudes_by_patient_N3 if len(l) > 1]

In [ ]:
print('STAGE N3\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE N3')
q1_freq_magnitude_interquartile_range_N3, q3_freq_magnitude_interquartile_range_N3 = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_N3 = [patients_freq_magnitudes_N3[i] for i in range(len(patients_freq_magnitudes_N3)) if np.quantile(freq_magnitudes_by_patient_N3[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N3[i], 0.25) < q1_freq_magnitude_interquartile_range_N3]
patients_in_q4_freq_magnitude_interquartile_range_N3 = [patients_freq_magnitudes_N3[i] for i in range(len(patients_freq_magnitudes_N3)) if np.quantile(freq_magnitudes_by_patient_N3[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N3[i], 0.25) > q3_freq_magnitude_interquartile_range_N3]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_N3 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_N3]
ahi_q4_freq_magnitude_interquartile_range_N3 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_N3]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_N3))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_N3, ahi_q4_freq_magnitude_interquartile_range_N3, alternative='greater'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_N3 = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_N3]
dessevs_q4_freq_magnitude_interquartile_range_N3 = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_N3]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_N3))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_N3, dessevs_q4_freq_magnitude_interquartile_range_N3, alternative='greater'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_N3 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N3]
lapses_q4_freq_magnitude_interquartile_range_N3 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N3]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_N3))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_N3, lapses_q4_freq_magnitude_interquartile_range_N3, alternative='less'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_N3 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N3]
speed_q4_freq_magnitude_interquartile_range_N3 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N3]
print(np.median(speed_q1_freq_magnitude_interquartile_range_N3))
print(np.median(speed_q4_freq_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_N3, speed_q4_freq_magnitude_interquartile_range_N3, alternative='less'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE N3')
q1_amp_magnitude_interquartile_range_N3, q3_amp_magnitude_interquartile_range_N3 = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_N3 = [patients_amp_magnitudes_N3[i] for i in range(len(patients_amp_magnitudes_N3)) if np.quantile(amp_magnitudes_by_patient_N3[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N3[i], 0.25) < q1_amp_magnitude_interquartile_range_N3]
patients_in_q4_amp_magnitude_interquartile_range_N3 = [patients_amp_magnitudes_N3[i] for i in range(len(patients_amp_magnitudes_N3)) if np.quantile(amp_magnitudes_by_patient_N3[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N3[i], 0.25) > q3_amp_magnitude_interquartile_range_N3]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_N3 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_N3]
ahi_q4_amp_magnitude_interquartile_range_N3 = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_N3]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_N3))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_N3, ahi_q4_amp_magnitude_interquartile_range_N3, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_N3 = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_N3]
dessevs_q4_amp_magnitude_interquartile_range_N3 = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_N3]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_N3))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_N3, dessevs_q4_amp_magnitude_interquartile_range_N3, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_N3 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N3]
lapses_q4_amp_magnitude_interquartile_range_N3 = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N3]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_N3))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_N3, lapses_q4_amp_magnitude_interquartile_range_N3, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_N3 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N3]
speed_q4_amp_magnitude_interquartile_range_N3 = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N3]
print(np.median(speed_q1_amp_magnitude_interquartile_range_N3))
print(np.median(speed_q4_amp_magnitude_interquartile_range_N3))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_N3, speed_q4_amp_magnitude_interquartile_range_N3, alternative='less'))

In [ ]:
def pvaluetext(pvalue):
    if pvalue >= 0.1:
        return '$p = %.2f$' % pvalue
    elif pvalue >= 0.01:
        return '$p = %.3f$' % pvalue
    else:
        text = '$p = %.1E$' % pvalue
        text = text[:text.index('E')] + '\\mathrm{E}' + text[text.index('E')+1:]
        return text

def pvalueaxtext(x, y, texty, ax):
    if np.median(x) < np.median(y):
        alternative = 'less'
    else:
        alternative = 'greater'
    pvalue = scipy.stats.mannwhitneyu(x, y, alternative=alternative).pvalue
    text = pvaluetext(pvalue)
    ax.text(1.5, texty, text, va='center', ha='center', **figfont8)
    if pvalue < 0.01:
        plt.setp(ax.spines.values(), lw=2)

### Normalization

In [ ]:
freq_magnitudes_by_patient_normalized = [[(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_normalized = [[(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]

In [ ]:
print('ALL STAGES\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_normalized]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - ALL STAGES')
q1_freq_magnitude_interquartile_range_all_stages_normalized, q3_freq_magnitude_interquartile_range_all_stages_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_normalized], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_all_stages_normalized = [patients[i] for i in range(len(patients)) if np.quantile(freq_magnitudes_by_patient_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_normalized[i], 0.25) < q1_freq_magnitude_interquartile_range_all_stages_normalized]
patients_in_q4_freq_magnitude_interquartile_range_all_stages_normalized = [patients[i] for i in range(len(patients)) if np.quantile(freq_magnitudes_by_patient_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_normalized[i], 0.25) > q3_freq_magnitude_interquartile_range_all_stages_normalized]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_all_stages_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages_normalized]
ahi_q4_freq_magnitude_interquartile_range_all_stages_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages_normalized]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_all_stages_normalized))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_all_stages_normalized, ahi_q4_freq_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_all_stages_normalized = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages_normalized]
dessevs_q4_freq_magnitude_interquartile_range_all_stages_normalized = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages_normalized]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_all_stages_normalized))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_all_stages_normalized, dessevs_q4_freq_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_all_stages_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages_normalized]
lapses_q4_freq_magnitude_interquartile_range_all_stages_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages_normalized]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_all_stages_normalized))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_all_stages_normalized, lapses_q4_freq_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_all_stages_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_all_stages_normalized]
speed_q4_freq_magnitude_interquartile_range_all_stages_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_all_stages_normalized]
print(np.median(speed_q1_freq_magnitude_interquartile_range_all_stages_normalized))
print(np.median(speed_q4_freq_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_all_stages_normalized, speed_q4_freq_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - ALL STAGES')
q1_amp_magnitude_interquartile_range_all_stages_normalized, q3_amp_magnitude_interquartile_range_all_stages_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_normalized], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_all_stages_normalized = [patients[i] for i in range(len(patients)) if np.quantile(amp_magnitudes_by_patient_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_normalized[i], 0.25) < q1_amp_magnitude_interquartile_range_all_stages_normalized]
patients_in_q4_amp_magnitude_interquartile_range_all_stages_normalized = [patients[i] for i in range(len(patients)) if np.quantile(amp_magnitudes_by_patient_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_normalized[i], 0.25) > q3_amp_magnitude_interquartile_range_all_stages_normalized]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_all_stages_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages_normalized]
ahi_q4_amp_magnitude_interquartile_range_all_stages_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages_normalized]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_all_stages_normalized))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_all_stages_normalized, ahi_q4_amp_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_all_stages_normalized = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages_normalized]
dessevs_q4_amp_magnitude_interquartile_range_all_stages_normalized = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages_normalized]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_all_stages_normalized))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_all_stages_normalized, dessevs_q4_amp_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_all_stages_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages_normalized]
lapses_q4_amp_magnitude_interquartile_range_all_stages_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages_normalized]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_all_stages_normalized))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_all_stages_normalized, lapses_q4_amp_magnitude_interquartile_range_all_stages_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_all_stages_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_all_stages_normalized]
speed_q4_amp_magnitude_interquartile_range_all_stages_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_all_stages_normalized]
print(np.median(speed_q1_amp_magnitude_interquartile_range_all_stages_normalized))
print(np.median(speed_q4_amp_magnitude_interquartile_range_all_stages_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_all_stages_normalized, speed_q4_amp_magnitude_interquartile_range_all_stages_normalized, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_R_normalized = [[(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_R_normalized = [l for l in freq_magnitudes_by_patient_R_normalized if len(l) > 1]
amp_magnitudes_by_patient_R_normalized = [[(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[0] and not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[0] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_R_normalized = [l for l in amp_magnitudes_by_patient_R_normalized if len(l) > 1]

In [ ]:
print('STAGE R\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R_normalized]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE R')
q1_freq_magnitude_interquartile_range_R_normalized, q3_freq_magnitude_interquartile_range_R_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R_normalized], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_R_normalized = [patients_freq_magnitudes_R[i] for i in range(len(patients_freq_magnitudes_R)) if np.quantile(freq_magnitudes_by_patient_R_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_R_normalized[i], 0.25) < q1_freq_magnitude_interquartile_range_R_normalized]
patients_in_q4_freq_magnitude_interquartile_range_R_normalized = [patients_freq_magnitudes_R[i] for i in range(len(patients_freq_magnitudes_R)) if np.quantile(freq_magnitudes_by_patient_R_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_R_normalized[i], 0.25) > q3_freq_magnitude_interquartile_range_R_normalized]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_R_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_R_normalized]
ahi_q4_freq_magnitude_interquartile_range_R_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_R_normalized]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_R_normalized))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_R_normalized, ahi_q4_freq_magnitude_interquartile_range_R_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_R_normalized = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_R_normalized]
dessevs_q4_freq_magnitude_interquartile_range_R_normalized = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_R_normalized]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_R_normalized))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_R_normalized, dessevs_q4_freq_magnitude_interquartile_range_R_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_R_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_R_normalized]
lapses_q4_freq_magnitude_interquartile_range_R_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_R_normalized]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_R_normalized))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_R_normalized, lapses_q4_freq_magnitude_interquartile_range_R_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_R_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_R_normalized]
speed_q4_freq_magnitude_interquartile_range_R_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_R_normalized]
print(np.median(speed_q1_freq_magnitude_interquartile_range_R_normalized))
print(np.median(speed_q4_freq_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_R_normalized, speed_q4_freq_magnitude_interquartile_range_R_normalized, alternative='less'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE R')
q1_amp_magnitude_interquartile_range_R_normalized, q3_amp_magnitude_interquartile_range_R_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R_normalized], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_R_normalized = [patients_amp_magnitudes_R[i] for i in range(len(patients_amp_magnitudes_R)) if np.quantile(amp_magnitudes_by_patient_R_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_R_normalized[i], 0.25) < q1_amp_magnitude_interquartile_range_R_normalized]
patients_in_q4_amp_magnitude_interquartile_range_R_normalized = [patients_amp_magnitudes_R[i] for i in range(len(patients_amp_magnitudes_R)) if np.quantile(amp_magnitudes_by_patient_R_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_R_normalized[i], 0.25) > q3_amp_magnitude_interquartile_range_R_normalized]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_R_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_R_normalized]
ahi_q4_amp_magnitude_interquartile_range_R_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_R_normalized]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_R_normalized))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_R_normalized, ahi_q4_amp_magnitude_interquartile_range_R_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_R_normalized = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_R_normalized]
dessevs_q4_amp_magnitude_interquartile_range_R_normalized = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_R_normalized]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_R_normalized))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_R_normalized, dessevs_q4_amp_magnitude_interquartile_range_R_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_R_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_R_normalized]
lapses_q4_amp_magnitude_interquartile_range_R_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_R_normalized]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_R_normalized))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_R_normalized, lapses_q4_amp_magnitude_interquartile_range_R_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_R_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_R_normalized]
speed_q4_amp_magnitude_interquartile_range_R_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_R_normalized]
print(np.median(speed_q1_amp_magnitude_interquartile_range_R_normalized))
print(np.median(speed_q4_amp_magnitude_interquartile_range_R_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_R_normalized, speed_q4_amp_magnitude_interquartile_range_R_normalized, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_N1_normalized = [[(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_N1_normalized = [l for l in freq_magnitudes_by_patient_N1_normalized if len(l) > 1]
amp_magnitudes_by_patient_N1_normalized = [[(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[1] and not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[1] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N1_normalized = [l for l in amp_magnitudes_by_patient_N1_normalized if len(l) > 1]

In [ ]:
print('STAGE N1\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1_normalized]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE N1')
q1_freq_magnitude_interquartile_range_N1_normalized, q3_freq_magnitude_interquartile_range_N1_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1_normalized], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_N1_normalized = [patients_freq_magnitudes_N1[i] for i in range(len(patients_freq_magnitudes_N1)) if np.quantile(freq_magnitudes_by_patient_N1_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N1_normalized[i], 0.25) < q1_freq_magnitude_interquartile_range_N1_normalized]
patients_in_q4_freq_magnitude_interquartile_range_N1_normalized = [patients_freq_magnitudes_N1[i] for i in range(len(patients_freq_magnitudes_N1)) if np.quantile(freq_magnitudes_by_patient_N1_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N1_normalized[i], 0.25) > q3_freq_magnitude_interquartile_range_N1_normalized]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_N1_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_N1_normalized]
ahi_q4_freq_magnitude_interquartile_range_N1_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_N1_normalized]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_N1_normalized))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_N1_normalized, ahi_q4_freq_magnitude_interquartile_range_N1_normalized, alternative='less'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_N1_normalized = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_N1_normalized]
dessevs_q4_freq_magnitude_interquartile_range_N1_normalized = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_N1_normalized]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_N1_normalized))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_N1_normalized, dessevs_q4_freq_magnitude_interquartile_range_N1_normalized, alternative='less'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_N1_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N1_normalized]
lapses_q4_freq_magnitude_interquartile_range_N1_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N1_normalized]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_N1_normalized))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_N1_normalized, lapses_q4_freq_magnitude_interquartile_range_N1_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_N1_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N1_normalized]
speed_q4_freq_magnitude_interquartile_range_N1_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N1_normalized]
print(np.median(speed_q1_freq_magnitude_interquartile_range_N1_normalized))
print(np.median(speed_q4_freq_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_N1_normalized, speed_q4_freq_magnitude_interquartile_range_N1_normalized, alternative='greater'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE N1')
q1_amp_magnitude_interquartile_range_N1_normalized, q3_amp_magnitude_interquartile_range_N1_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1_normalized], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_N1_normalized = [patients_amp_magnitudes_N1[i] for i in range(len(patients_amp_magnitudes_N1)) if np.quantile(amp_magnitudes_by_patient_N1_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N1_normalized[i], 0.25) < q1_amp_magnitude_interquartile_range_N1_normalized]
patients_in_q4_amp_magnitude_interquartile_range_N1_normalized = [patients_amp_magnitudes_N1[i] for i in range(len(patients_amp_magnitudes_N1)) if np.quantile(amp_magnitudes_by_patient_N1_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N1_normalized[i], 0.25) > q3_amp_magnitude_interquartile_range_N1_normalized]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_N1_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_N1_normalized]
ahi_q4_amp_magnitude_interquartile_range_N1_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_N1_normalized]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_N1_normalized))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_N1_normalized, ahi_q4_amp_magnitude_interquartile_range_N1_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_N1_normalized = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_N1_normalized]
dessevs_q4_amp_magnitude_interquartile_range_N1_normalized = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_N1_normalized]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_N1_normalized))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_N1_normalized, dessevs_q4_amp_magnitude_interquartile_range_N1_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_N1_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N1_normalized]
lapses_q4_amp_magnitude_interquartile_range_N1_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N1_normalized]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_N1_normalized))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_N1_normalized, lapses_q4_amp_magnitude_interquartile_range_N1_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_N1_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N1_normalized]
speed_q4_amp_magnitude_interquartile_range_N1_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N1_normalized]
print(np.median(speed_q1_amp_magnitude_interquartile_range_N1_normalized))
print(np.median(speed_q4_amp_magnitude_interquartile_range_N1_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_N1_normalized, speed_q4_amp_magnitude_interquartile_range_N1_normalized, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_N2_normalized = [[(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_N2_normalized = [l for l in freq_magnitudes_by_patient_N2_normalized if len(l) > 1]
amp_magnitudes_by_patient_N2_normalized = [[(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[2] and not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[2] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N2_normalized = [l for l in amp_magnitudes_by_patient_N2_normalized if len(l) > 1]

In [ ]:
print('STAGE N2\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2_normalized]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE N2')
q1_freq_magnitude_interquartile_range_N2_normalized, q3_freq_magnitude_interquartile_range_N2_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2_normalized], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_N2_normalized = [patients_freq_magnitudes_N2[i] for i in range(len(patients_freq_magnitudes_N2)) if np.quantile(freq_magnitudes_by_patient_N2_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N2_normalized[i], 0.25) < q1_freq_magnitude_interquartile_range_N2_normalized]
patients_in_q4_freq_magnitude_interquartile_range_N2_normalized = [patients_freq_magnitudes_N2[i] for i in range(len(patients_freq_magnitudes_N2)) if np.quantile(freq_magnitudes_by_patient_N2_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N2_normalized[i], 0.25) > q3_freq_magnitude_interquartile_range_N2_normalized]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_N2_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_N2_normalized]
ahi_q4_freq_magnitude_interquartile_range_N2_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_N2_normalized]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_N2_normalized))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_N2_normalized, ahi_q4_freq_magnitude_interquartile_range_N2_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_N2_normalized = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_N2_normalized]
dessevs_q4_freq_magnitude_interquartile_range_N2_normalized = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_N2_normalized]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_N2_normalized))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_N2_normalized, dessevs_q4_freq_magnitude_interquartile_range_N2_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_N2_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N2_normalized]
lapses_q4_freq_magnitude_interquartile_range_N2_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N2_normalized]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_N2_normalized))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_N2_normalized, lapses_q4_freq_magnitude_interquartile_range_N2_normalized, alternative='less'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_N2_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N2_normalized]
speed_q4_freq_magnitude_interquartile_range_N2_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N2_normalized]
print(np.median(speed_q1_freq_magnitude_interquartile_range_N2_normalized))
print(np.median(speed_q4_freq_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_N2_normalized, speed_q4_freq_magnitude_interquartile_range_N2_normalized, alternative='greater'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE N2')
q1_amp_magnitude_interquartile_range_N2_normalized, q3_amp_magnitude_interquartile_range_N2_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2_normalized], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_N2_normalized = [patients_amp_magnitudes_N2[i] for i in range(len(patients_amp_magnitudes_N2)) if np.quantile(amp_magnitudes_by_patient_N2_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N2_normalized[i], 0.25) < q1_amp_magnitude_interquartile_range_N2_normalized]
patients_in_q4_amp_magnitude_interquartile_range_N2_normalized = [patients_amp_magnitudes_N2[i] for i in range(len(patients_amp_magnitudes_N2)) if np.quantile(amp_magnitudes_by_patient_N2_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N2_normalized[i], 0.25) > q3_amp_magnitude_interquartile_range_N2_normalized]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_N2_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_N2_normalized]
ahi_q4_amp_magnitude_interquartile_range_N2_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_N2_normalized]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_N2_normalized))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_N2_normalized, ahi_q4_amp_magnitude_interquartile_range_N2_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_N2_normalized = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_N2_normalized]
dessevs_q4_amp_magnitude_interquartile_range_N2_normalized = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_N2_normalized]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_N2_normalized))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_N2_normalized, dessevs_q4_amp_magnitude_interquartile_range_N2_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_N2_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N2_normalized]
lapses_q4_amp_magnitude_interquartile_range_N2_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N2_normalized]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_N2_normalized))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_N2_normalized, lapses_q4_amp_magnitude_interquartile_range_N2_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_N2_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N2_normalized]
speed_q4_amp_magnitude_interquartile_range_N2_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N2_normalized]
print(np.median(speed_q1_amp_magnitude_interquartile_range_N2_normalized))
print(np.median(speed_q4_amp_magnitude_interquartile_range_N2_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_N2_normalized, speed_q4_amp_magnitude_interquartile_range_N2_normalized, alternative='less'))

In [ ]:
freq_magnitudes_by_patient_N3_normalized = [[(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(freq_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
freq_magnitudes_by_patient_N3_normalized = [l for l in freq_magnitudes_by_patient_N3_normalized if len(l) > 1]
amp_magnitudes_by_patient_N3_normalized = [[(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if j in indices_by_stage[3] and not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if j in indices_by_stage_spont[3] and not np.isnan(amp_startbaselines_spont[j])] for i in tqdm(range(len(patients)))]
amp_magnitudes_by_patient_N3_normalized = [l for l in amp_magnitudes_by_patient_N3_normalized if len(l) > 1]

In [ ]:
print('STAGE N3\n')
print('Frequency magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3_normalized]))
print('Amplitude magnitudes:')
print('  - Interquartile range of all arousals:', np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.25))
print('  - Median of patientwise interquartile ranges:', np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3_normalized]))

In [ ]:
print('FREQUENCY MAGNITUDE VARIATION - STAGE N3')
q1_freq_magnitude_interquartile_range_N3_normalized, q3_freq_magnitude_interquartile_range_N3_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3_normalized], [25, 75])
patients_in_q1_freq_magnitude_interquartile_range_N3_normalized = [patients_freq_magnitudes_N3[i] for i in range(len(patients_freq_magnitudes_N3)) if np.quantile(freq_magnitudes_by_patient_N3_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N3_normalized[i], 0.25) < q1_freq_magnitude_interquartile_range_N3_normalized]
patients_in_q4_freq_magnitude_interquartile_range_N3_normalized = [patients_freq_magnitudes_N3[i] for i in range(len(patients_freq_magnitudes_N3)) if np.quantile(freq_magnitudes_by_patient_N3_normalized[i], 0.75) - np.quantile(freq_magnitudes_by_patient_N3_normalized[i], 0.25) > q3_freq_magnitude_interquartile_range_N3_normalized]
print('AHI:')
ahi_q1_freq_magnitude_interquartile_range_N3_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_freq_magnitude_interquartile_range_N3_normalized]
ahi_q4_freq_magnitude_interquartile_range_N3_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_freq_magnitude_interquartile_range_N3_normalized]
print(np.median(ahi_q1_freq_magnitude_interquartile_range_N3_normalized))
print(np.median(ahi_q4_freq_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_freq_magnitude_interquartile_range_N3_normalized, ahi_q4_freq_magnitude_interquartile_range_N3_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_freq_magnitude_interquartile_range_N3_normalized = [dessevs[patient] for patient in patients_in_q1_freq_magnitude_interquartile_range_N3_normalized]
dessevs_q4_freq_magnitude_interquartile_range_N3_normalized = [dessevs[patient] for patient in patients_in_q4_freq_magnitude_interquartile_range_N3_normalized]
print(np.median(dessevs_q1_freq_magnitude_interquartile_range_N3_normalized))
print(np.median(dessevs_q4_freq_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_freq_magnitude_interquartile_range_N3_normalized, dessevs_q4_freq_magnitude_interquartile_range_N3_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_freq_magnitude_interquartile_range_N3_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N3_normalized]
lapses_q4_freq_magnitude_interquartile_range_N3_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N3_normalized]
print(np.median(lapses_q1_freq_magnitude_interquartile_range_N3_normalized))
print(np.median(lapses_q4_freq_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_freq_magnitude_interquartile_range_N3_normalized, lapses_q4_freq_magnitude_interquartile_range_N3_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_freq_magnitude_interquartile_range_N3_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_freq_magnitude_interquartile_range_N3_normalized]
speed_q4_freq_magnitude_interquartile_range_N3_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_freq_magnitude_interquartile_range_N3_normalized]
print(np.median(speed_q1_freq_magnitude_interquartile_range_N3_normalized))
print(np.median(speed_q4_freq_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_freq_magnitude_interquartile_range_N3_normalized, speed_q4_freq_magnitude_interquartile_range_N3_normalized, alternative='less'))

In [ ]:
print('AMPLITUDE MAGNITUDE VARIATION - STAGE N3')
q1_amp_magnitude_interquartile_range_N3_normalized, q3_amp_magnitude_interquartile_range_N3_normalized = np.percentile([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3_normalized], [25, 75])
patients_in_q1_amp_magnitude_interquartile_range_N3_normalized = [patients_amp_magnitudes_N3[i] for i in range(len(patients_amp_magnitudes_N3)) if np.quantile(amp_magnitudes_by_patient_N3_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N3_normalized[i], 0.25) < q1_amp_magnitude_interquartile_range_N3_normalized]
patients_in_q4_amp_magnitude_interquartile_range_N3_normalized = [patients_amp_magnitudes_N3[i] for i in range(len(patients_amp_magnitudes_N3)) if np.quantile(amp_magnitudes_by_patient_N3_normalized[i], 0.75) - np.quantile(amp_magnitudes_by_patient_N3_normalized[i], 0.25) > q3_amp_magnitude_interquartile_range_N3_normalized]
print('AHI:')
ahi_q1_amp_magnitude_interquartile_range_N3_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q1_amp_magnitude_interquartile_range_N3_normalized]
ahi_q4_amp_magnitude_interquartile_range_N3_normalized = [(len(apnea_inds[patient])+len(central_apnea_inds[patient])+len(mixed_apnea_inds[patient])+len(hypopnea_inds[patient]))/(len([stage for stage in stages[patient] if stage != 0])*30/3600) for patient in patients_in_q4_amp_magnitude_interquartile_range_N3_normalized]
print(np.median(ahi_q1_amp_magnitude_interquartile_range_N3_normalized))
print(np.median(ahi_q4_amp_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(ahi_q1_amp_magnitude_interquartile_range_N3_normalized, ahi_q4_amp_magnitude_interquartile_range_N3_normalized, alternative='greater'))
print('DesSev:')
dessevs_q1_amp_magnitude_interquartile_range_N3_normalized = [dessevs[patient] for patient in patients_in_q1_amp_magnitude_interquartile_range_N3_normalized]
dessevs_q4_amp_magnitude_interquartile_range_N3_normalized = [dessevs[patient] for patient in patients_in_q4_amp_magnitude_interquartile_range_N3_normalized]
print(np.median(dessevs_q1_amp_magnitude_interquartile_range_N3_normalized))
print(np.median(dessevs_q4_amp_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(dessevs_q1_amp_magnitude_interquartile_range_N3_normalized, dessevs_q4_amp_magnitude_interquartile_range_N3_normalized, alternative='greater'))
print('PVT lapses:')
lapses_q1_amp_magnitude_interquartile_range_N3_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N3_normalized]
lapses_q4_amp_magnitude_interquartile_range_N3_normalized = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N3_normalized]
print(np.median(lapses_q1_amp_magnitude_interquartile_range_N3_normalized))
print(np.median(lapses_q4_amp_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(lapses_q1_amp_magnitude_interquartile_range_N3_normalized, lapses_q4_amp_magnitude_interquartile_range_N3_normalized, alternative='greater'))
print('PVT mean speed:')
speed_q1_amp_magnitude_interquartile_range_N3_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q1_amp_magnitude_interquartile_range_N3_normalized]
speed_q4_amp_magnitude_interquartile_range_N3_normalized = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients_in_q4_amp_magnitude_interquartile_range_N3_normalized]
print(np.median(speed_q1_amp_magnitude_interquartile_range_N3_normalized))
print(np.median(speed_q4_amp_magnitude_interquartile_range_N3_normalized))
print(scipy.stats.mannwhitneyu(speed_q1_amp_magnitude_interquartile_range_N3_normalized, speed_q4_amp_magnitude_interquartile_range_N3_normalized, alternative='less'))

### Comparison between absolute and normalized

In [ ]:
freq_relative_variations = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient]) / (np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R]) / (np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1]) / (np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2]) / (np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3]) / (np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3 for val in l], 0.25))
]
print(freq_relative_variations)

freq_relative_variations_normalized = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_R_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_R_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N1_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N1_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N2_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N2_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in freq_magnitudes_by_patient_N3_normalized]) / (np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in freq_magnitudes_by_patient_N3_normalized for val in l], 0.25))
]
print(freq_relative_variations_normalized)

print(scipy.stats.wilcoxon(freq_relative_variations, freq_relative_variations_normalized).pvalue)

In [ ]:
amp_relative_variations = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient]) / (np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R]) / (np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1]) / (np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2]) / (np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2 for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3]) / (np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3 for val in l], 0.25))
]
print(amp_relative_variations)

amp_relative_variations_normalized = [
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_R_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_R_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N1_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N1_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N2_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N2_normalized for val in l], 0.25)),
    np.median([np.quantile(l, 0.75) - np.quantile(l, 0.25) for l in amp_magnitudes_by_patient_N3_normalized]) / (np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.75) - np.quantile([val for l in amp_magnitudes_by_patient_N3_normalized for val in l], 0.25))
]
print(amp_relative_variations_normalized)

print(scipy.stats.wilcoxon(amp_relative_variations, amp_relative_variations_normalized).pvalue)

## PVT number of lapses quartile comparison

In [ ]:
median_freq_magnitude_by_patient = [np.median([freq_eventual_expected_changes[j][0] - freq_startbaselines[j] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(freq_startbaselines[j])] + [freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(freq_startbaselines_spont[j])]) for i in tqdm(range(len(patients)))]
median_amp_magnitude_by_patient = [np.median([-amp_eventual_expected_changes[j][0] + amp_startbaselines[j] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(amp_startbaselines[j])] + [-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(amp_startbaselines_spont[j])]) for i in tqdm(range(len(patients)))]

In [ ]:
q1_freq_magnitude, q3_freq_magnitude = np.percentile(median_freq_magnitude_by_patient, [25, 75])

In [ ]:
patient_inds_in_q1_freq_magnitude = [i for i in range(len(patients)) if median_freq_magnitude_by_patient[i] < q1_freq_magnitude]
patient_inds_in_q4_freq_magnitude = [i for i in range(len(patients)) if median_freq_magnitude_by_patient[i] > q3_freq_magnitude]

In [ ]:
number_of_lapses_by_patient = [len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients]

In [ ]:
number_of_lapses_in_q1_freq_magnitude = [number_of_lapses_by_patient[i] for i in patient_inds_in_q1_freq_magnitude]
number_of_lapses_in_q4_freq_magnitude = [number_of_lapses_by_patient[i] for i in patient_inds_in_q4_freq_magnitude]
print('Q1:', np.median(number_of_lapses_in_q1_freq_magnitude))
print('Q4:', np.median(number_of_lapses_in_q4_freq_magnitude))
print(scipy.stats.mannwhitneyu(number_of_lapses_in_q1_freq_magnitude, number_of_lapses_in_q4_freq_magnitude, alternative='greater'))

In [ ]:
q1_amp_magnitude, q3_amp_magnitude = np.percentile(median_amp_magnitude_by_patient, [25, 75])

In [ ]:
patient_inds_in_q1_amp_magnitude = [i for i in range(len(patients)) if median_amp_magnitude_by_patient[i] < q1_amp_magnitude]
patient_inds_in_q4_amp_magnitude = [i for i in range(len(patients)) if median_amp_magnitude_by_patient[i] > q3_amp_magnitude]

In [ ]:
number_of_lapses_in_q1_amp_magnitude = [number_of_lapses_by_patient[i] for i in patient_inds_in_q1_amp_magnitude]
number_of_lapses_in_q4_amp_magnitude = [number_of_lapses_by_patient[i] for i in patient_inds_in_q4_amp_magnitude]
print('Q1:', np.median(number_of_lapses_in_q1_amp_magnitude))
print('Q4:', np.median(number_of_lapses_in_q4_amp_magnitude))
print(scipy.stats.mannwhitneyu(number_of_lapses_in_q1_amp_magnitude, number_of_lapses_in_q4_amp_magnitude, alternative='greater'))

### Normalization

In [ ]:
median_freq_magnitude_by_patient_normalized = [np.median([(freq_eventual_expected_changes[j][0] - freq_startbaselines[j]) / median_freq_magnitude_by_type[arousal_types[j]] if not np.isnan(freq_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(freq_startbaselines[j])] + [(freq_eventual_expected_changes_spont[j][0] - freq_startbaselines_spont[j]) / median_freq_magnitude_by_type[4] if not np.isnan(freq_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(freq_startbaselines_spont[j])]) for i in tqdm(range(len(patients)))]
median_amp_magnitude_by_patient_normalized = [np.median([(-amp_eventual_expected_changes[j][0] + amp_startbaselines[j]) / median_amp_magnitude_by_type[arousal_types[j]] if not np.isnan(amp_eventual_expected_changes[j][0]) else 0 for j in indices_by_patient[i] if not np.isnan(amp_startbaselines[j])] + [(-amp_eventual_expected_changes_spont[j][0] + amp_startbaselines_spont[j]) / median_amp_magnitude_by_type[4] if not np.isnan(amp_eventual_expected_changes_spont[j][0]) else 0 for j in indices_by_patient_spont[i] if not np.isnan(amp_startbaselines_spont[j])]) for i in tqdm(range(len(patients)))]

In [ ]:
q1_freq_magnitude_normalized, q3_freq_magnitude_normalized = np.percentile(median_freq_magnitude_by_patient_normalized, [25, 75])

In [ ]:
patient_inds_in_q1_freq_magnitude_normalized = [i for i in range(len(patients)) if median_freq_magnitude_by_patient_normalized[i] < q1_freq_magnitude_normalized]
patient_inds_in_q4_freq_magnitude_normalized = [i for i in range(len(patients)) if median_freq_magnitude_by_patient_normalized[i] > q3_freq_magnitude_normalized]

In [ ]:
number_of_lapses_in_q1_freq_magnitude_normalized = [number_of_lapses_by_patient[i] for i in patient_inds_in_q1_freq_magnitude_normalized]
number_of_lapses_in_q4_freq_magnitude_normalized = [number_of_lapses_by_patient[i] for i in patient_inds_in_q4_freq_magnitude_normalized]
print('Q1:', np.median(number_of_lapses_in_q1_freq_magnitude_normalized))
print('Q4:', np.median(number_of_lapses_in_q4_freq_magnitude_normalized))
print(scipy.stats.mannwhitneyu(number_of_lapses_in_q1_freq_magnitude_normalized, number_of_lapses_in_q4_freq_magnitude_normalized, alternative='greater'))

In [ ]:
q1_amp_magnitude_normalized, q3_amp_magnitude_normalized = np.percentile(median_amp_magnitude_by_patient_normalized, [25, 75])

In [ ]:
patient_inds_in_q1_amp_magnitude_normalized = [i for i in range(len(patients)) if median_amp_magnitude_by_patient_normalized[i] < q1_amp_magnitude_normalized]
patient_inds_in_q4_amp_magnitude_normalized = [i for i in range(len(patients)) if median_amp_magnitude_by_patient_normalized[i] > q3_amp_magnitude_normalized]

In [ ]:
number_of_lapses_in_q1_amp_magnitude_normalized = [number_of_lapses_by_patient[i] for i in patient_inds_in_q1_amp_magnitude_normalized]
number_of_lapses_in_q4_amp_magnitude_normalized = [number_of_lapses_by_patient[i] for i in patient_inds_in_q4_amp_magnitude_normalized]
print('Q1:', np.median(number_of_lapses_in_q1_amp_magnitude_normalized))
print('Q4:', np.median(number_of_lapses_in_q4_amp_magnitude_normalized))
print(scipy.stats.mannwhitneyu(number_of_lapses_in_q1_amp_magnitude_normalized, number_of_lapses_in_q4_amp_magnitude_normalized, alternative='greater'))

## PVT mean speed quartile comparison

In [ ]:
mean_speed_by_patient = [np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients]

In [ ]:
mean_speed_in_q1_freq_magnitude = [mean_speed_by_patient[i] for i in patient_inds_in_q1_freq_magnitude]
mean_speed_in_q4_freq_magnitude = [mean_speed_by_patient[i] for i in patient_inds_in_q4_freq_magnitude]
print('Q1:', np.median(mean_speed_in_q1_freq_magnitude))
print('Q4:', np.median(mean_speed_in_q4_freq_magnitude))
print(scipy.stats.mannwhitneyu(mean_speed_in_q1_freq_magnitude, mean_speed_in_q4_freq_magnitude, alternative='less'))

In [ ]:
mean_speed_in_q1_amp_magnitude = [mean_speed_by_patient[i] for i in patient_inds_in_q1_amp_magnitude]
mean_speed_in_q4_amp_magnitude = [mean_speed_by_patient[i] for i in patient_inds_in_q4_amp_magnitude]
print('Q1:', np.median(mean_speed_in_q1_amp_magnitude))
print('Q4:', np.median(mean_speed_in_q4_amp_magnitude))
print(scipy.stats.mannwhitneyu(mean_speed_in_q1_amp_magnitude, mean_speed_in_q4_amp_magnitude, alternative='less'))

### Normalization

In [ ]:
mean_speed_in_q1_freq_magnitude_normalized = [mean_speed_by_patient[i] for i in patient_inds_in_q1_freq_magnitude_normalized]
mean_speed_in_q4_freq_magnitude_normalized = [mean_speed_by_patient[i] for i in patient_inds_in_q4_freq_magnitude_normalized]
print('Q1:', np.median(mean_speed_in_q1_freq_magnitude_normalized))
print('Q4:', np.median(mean_speed_in_q4_freq_magnitude_normalized))
print(scipy.stats.mannwhitneyu(mean_speed_in_q1_freq_magnitude_normalized, mean_speed_in_q4_freq_magnitude_normalized, alternative='less'))

In [ ]:
mean_speed_in_q1_amp_magnitude_normalized = [mean_speed_by_patient[i] for i in patient_inds_in_q1_amp_magnitude_normalized]
mean_speed_in_q4_amp_magnitude_normalized = [mean_speed_by_patient[i] for i in patient_inds_in_q4_amp_magnitude_normalized]
print('Q1:', np.median(mean_speed_in_q1_amp_magnitude_normalized))
print('Q4:', np.median(mean_speed_in_q4_amp_magnitude_normalized))
print(scipy.stats.mannwhitneyu(mean_speed_in_q1_amp_magnitude_normalized, mean_speed_in_q4_amp_magnitude_normalized, alternative='less'))

## AHI quartile comparison

In [ ]:
AHI_in_q1_freq_magnitude = [ahi[i] for i in patient_inds_in_q1_freq_magnitude]
AHI_in_q4_freq_magnitude = [ahi[i] for i in patient_inds_in_q4_freq_magnitude]
print('Q1:', np.median(AHI_in_q1_freq_magnitude))
print('Q4:', np.median(AHI_in_q4_freq_magnitude))
print(scipy.stats.mannwhitneyu(AHI_in_q1_freq_magnitude, AHI_in_q4_freq_magnitude, alternative='less'))

In [ ]:
AHI_in_q1_amp_magnitude = [ahi[i] for i in patient_inds_in_q1_amp_magnitude]
AHI_in_q4_amp_magnitude = [ahi[i] for i in patient_inds_in_q4_amp_magnitude]
print('Q1:', np.median(AHI_in_q1_amp_magnitude))
print('Q4:', np.median(AHI_in_q4_amp_magnitude))
print(scipy.stats.mannwhitneyu(AHI_in_q1_amp_magnitude, AHI_in_q4_amp_magnitude, alternative='greater'))

### Normalization

In [ ]:
AHI_in_q1_freq_magnitude_normalized = [ahi[i] for i in patient_inds_in_q1_freq_magnitude_normalized]
AHI_in_q4_freq_magnitude_normalized = [ahi[i] for i in patient_inds_in_q4_freq_magnitude_normalized]
print('Q1:', np.median(AHI_in_q1_freq_magnitude_normalized))
print('Q4:', np.median(AHI_in_q4_freq_magnitude_normalized))
print(scipy.stats.mannwhitneyu(AHI_in_q1_freq_magnitude_normalized, AHI_in_q4_freq_magnitude_normalized, alternative='less'))

In [ ]:
AHI_in_q1_amp_magnitude_normalized = [ahi[i] for i in patient_inds_in_q1_amp_magnitude_normalized]
AHI_in_q4_amp_magnitude_normalized = [ahi[i] for i in patient_inds_in_q4_amp_magnitude_normalized]
print('Q1:', np.median(AHI_in_q1_amp_magnitude_normalized))
print('Q4:', np.median(AHI_in_q4_amp_magnitude_normalized))
print(scipy.stats.mannwhitneyu(AHI_in_q1_amp_magnitude_normalized, AHI_in_q4_amp_magnitude_normalized, alternative='greater'))

## DesSev quartile comparison

In [ ]:
dessev_in_q1_freq_magnitude = [dessevs[patients[i]] for i in patient_inds_in_q1_freq_magnitude]
dessev_in_q4_freq_magnitude = [dessevs[patients[i]] for i in patient_inds_in_q4_freq_magnitude]
print('Q1:', np.median(dessev_in_q1_freq_magnitude))
print('Q4:', np.median(dessev_in_q4_freq_magnitude))
print(scipy.stats.mannwhitneyu(dessev_in_q1_freq_magnitude, dessev_in_q4_freq_magnitude, alternative='less'))

In [ ]:
dessev_in_q1_amp_magnitude = [dessevs[patients[i]] for i in patient_inds_in_q1_amp_magnitude]
dessev_in_q4_amp_magnitude = [dessevs[patients[i]] for i in patient_inds_in_q4_amp_magnitude]
print('Q1:', np.median(dessev_in_q1_amp_magnitude))
print('Q4:', np.median(dessev_in_q4_amp_magnitude))
print(scipy.stats.mannwhitneyu(dessev_in_q1_amp_magnitude, dessev_in_q4_amp_magnitude, alternative='greater'))

### Normalization

In [ ]:
dessev_in_q1_freq_magnitude_normalized = [dessevs[patients[i]] for i in patient_inds_in_q1_freq_magnitude_normalized]
dessev_in_q4_freq_magnitude_normalized = [dessevs[patients[i]] for i in patient_inds_in_q4_freq_magnitude_normalized]
print('Q1:', np.median(dessev_in_q1_freq_magnitude_normalized))
print('Q4:', np.median(dessev_in_q4_freq_magnitude_normalized))
print(scipy.stats.mannwhitneyu(dessev_in_q1_freq_magnitude_normalized, dessev_in_q4_freq_magnitude_normalized, alternative='less'))

In [ ]:
dessev_in_q1_amp_magnitude_normalized = [dessevs[patients[i]] for i in patient_inds_in_q1_amp_magnitude_normalized]
dessev_in_q4_amp_magnitude_normalized = [dessevs[patients[i]] for i in patient_inds_in_q4_amp_magnitude_normalized]
print('Q1:', np.median(dessev_in_q1_amp_magnitude_normalized))
print('Q4:', np.median(dessev_in_q4_amp_magnitude_normalized))
print(scipy.stats.mannwhitneyu(dessev_in_q1_amp_magnitude_normalized, dessev_in_q4_amp_magnitude_normalized, alternative='greater'))

## Quartile comparisons figure

In [ ]:
if 8143 in plt.get_fignums(): plt.close(8143)
fig, axs = plt.subplots(nrows=4, ncols=4, sharex=True, sharey='row', num=8143, figsize=(17/2.54, 6), dpi=1000)

axs[0,0].violinplot([AHI_in_q1_freq_magnitude, AHI_in_q4_freq_magnitude], showextrema=False, showmedians=True)
axs[0,1].violinplot([AHI_in_q1_freq_magnitude_normalized, AHI_in_q4_freq_magnitude_normalized], showextrema=False, showmedians=True)
axs[0,2].violinplot([AHI_in_q1_amp_magnitude, AHI_in_q4_amp_magnitude], showextrema=False, showmedians=True)
axs[0,3].violinplot([AHI_in_q1_amp_magnitude_normalized, AHI_in_q4_amp_magnitude_normalized], showextrema=False, showmedians=True)

axs[0,0].set_ylim(0, np.percentile(ahi, 90))
texty = 0.9*np.percentile(ahi, 90)
pvalueaxtext(AHI_in_q1_freq_magnitude, AHI_in_q4_freq_magnitude, texty, axs[0,0])
pvalueaxtext(AHI_in_q1_freq_magnitude_normalized, AHI_in_q4_freq_magnitude_normalized, texty, axs[0,1])
pvalueaxtext(AHI_in_q1_amp_magnitude, AHI_in_q4_amp_magnitude, texty, axs[0,2])
pvalueaxtext(AHI_in_q1_amp_magnitude_normalized, AHI_in_q4_amp_magnitude_normalized, texty, axs[0,3])

axs[1,0].violinplot([dessev_in_q1_freq_magnitude, dessev_in_q4_freq_magnitude], showextrema=False, showmedians=True)
axs[1,1].violinplot([dessev_in_q1_freq_magnitude_normalized, dessev_in_q4_freq_magnitude_normalized], showextrema=False, showmedians=True)
axs[1,2].violinplot([dessev_in_q1_amp_magnitude, dessev_in_q4_amp_magnitude], showextrema=False, showmedians=True)
axs[1,3].violinplot([dessev_in_q1_amp_magnitude_normalized, dessev_in_q4_amp_magnitude_normalized], showextrema=False, showmedians=True)

axs[1,0].set_ylim(0, np.percentile(list(dessevs.values()), 90))
texty = 0.9*np.percentile(list(dessevs.values()), 90)
pvalueaxtext(dessev_in_q1_freq_magnitude, dessev_in_q4_freq_magnitude, texty, axs[1,0])
pvalueaxtext(dessev_in_q1_freq_magnitude_normalized, dessev_in_q4_freq_magnitude_normalized, texty, axs[1,1])
pvalueaxtext(dessev_in_q1_amp_magnitude, dessev_in_q4_amp_magnitude, texty, axs[1,2])
pvalueaxtext(dessev_in_q1_amp_magnitude_normalized, dessev_in_q4_amp_magnitude_normalized, texty, axs[1,3])

axs[2,0].violinplot([number_of_lapses_in_q1_freq_magnitude, number_of_lapses_in_q4_freq_magnitude], showextrema=False, showmedians=True)
axs[2,1].violinplot([number_of_lapses_in_q1_freq_magnitude_normalized, number_of_lapses_in_q4_freq_magnitude_normalized], showextrema=False, showmedians=True)
axs[2,2].violinplot([number_of_lapses_in_q1_amp_magnitude, number_of_lapses_in_q4_amp_magnitude], showextrema=False, showmedians=True)
axs[2,3].violinplot([number_of_lapses_in_q1_amp_magnitude_normalized, number_of_lapses_in_q4_amp_magnitude_normalized], showextrema=False, showmedians=True)

axs[2,0].set_ylim(0, np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90))
texty = 0.9*np.percentile([len([rt for rt in pvt_rts_by_patient[patient] if rt > 500]) for patient in patients], 90)
pvalueaxtext(number_of_lapses_in_q1_freq_magnitude, number_of_lapses_in_q4_freq_magnitude, texty, axs[2,0])
pvalueaxtext(number_of_lapses_in_q1_freq_magnitude_normalized, number_of_lapses_in_q4_freq_magnitude_normalized, texty, axs[2,1])
pvalueaxtext(number_of_lapses_in_q1_amp_magnitude, number_of_lapses_in_q4_amp_magnitude, texty, axs[2,2])
pvalueaxtext(number_of_lapses_in_q1_amp_magnitude_normalized, number_of_lapses_in_q4_amp_magnitude_normalized, texty, axs[2,3])

axs[3,0].violinplot([mean_speed_in_q1_freq_magnitude, mean_speed_in_q4_freq_magnitude], showextrema=False, showmedians=True)
axs[3,1].violinplot([mean_speed_in_q1_freq_magnitude_normalized, mean_speed_in_q4_freq_magnitude_normalized], showextrema=False, showmedians=True)
axs[3,2].violinplot([mean_speed_in_q1_amp_magnitude, mean_speed_in_q4_amp_magnitude], showextrema=False, showmedians=True)
axs[3,3].violinplot([mean_speed_in_q1_amp_magnitude_normalized, mean_speed_in_q4_amp_magnitude_normalized], showextrema=False, showmedians=True)

axs[3,0].set_ylim(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10), np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90))
texty = np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10) + 0.9*(np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 90) - np.percentile([np.mean([1000/rt for rt in pvt_rts_by_patient[patient] if rt != 0]) for patient in patients], 10))
pvalueaxtext(mean_speed_in_q1_freq_magnitude, mean_speed_in_q4_freq_magnitude, texty, axs[3,0])
pvalueaxtext(mean_speed_in_q1_freq_magnitude_normalized, mean_speed_in_q4_freq_magnitude_normalized, texty, axs[3,1])
pvalueaxtext(mean_speed_in_q1_amp_magnitude, mean_speed_in_q4_amp_magnitude, texty, axs[3,2])
pvalueaxtext(mean_speed_in_q1_amp_magnitude_normalized, mean_speed_in_q4_amp_magnitude_normalized, texty, axs[3,3])

axs[0,0].set_title('Absolute frequency \nresponse', **figfont10)
axs[0,1].set_title('Normalized frequency \nresponse', **figfont10)
axs[0,2].set_title('Absolute amplitude \nresponse', **figfont10)
axs[0,3].set_title('Normalized amplitude \nresponse', **figfont10)

axs[0,0].set_ylabel('AHI', **figfont10)
axs[1,0].set_ylabel('DesSev', **figfont10)
axs[2,0].set_ylabel('PVT lapses', **figfont10)
axs[3,0].set_ylabel('PVT mean speed', **figfont10)

axs[0,0].set_xlabel('(a)', **figfont8)
axs[0,1].set_xlabel('(b)', **figfont8)
axs[0,2].set_xlabel('(c)', **figfont8)
axs[0,3].set_xlabel('(d)', **figfont8)
axs[1,0].set_xlabel('(e)', **figfont8)
axs[1,1].set_xlabel('(f)', **figfont8)
axs[1,2].set_xlabel('(g)', **figfont8)
axs[1,3].set_xlabel('(h)', **figfont8)
axs[2,0].set_xlabel('(i)', **figfont8)
axs[2,1].set_xlabel('(j)', **figfont8)
axs[2,2].set_xlabel('(k)', **figfont8)
axs[2,3].set_xlabel('(l)', **figfont8)
axs[3,0].set_xlabel('(m)', **figfont8)
axs[3,1].set_xlabel('(n)', **figfont8)
axs[3,2].set_xlabel('(o)', **figfont8)
axs[3,3].set_xlabel('(p)', **figfont8)

axs[0,0].set_xticks([1, 2])
axs[0,0].set_xticklabels(['Q1', 'Q4'], **figfont10)

fig.align_labels()
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('fig3.png')